In [1]:
# Parameters
run_date = "2026-01-01"  # papermill replacement
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
config_path = os.environ.get("DATUM_API_CONFIG_PATH", "../ops/datum_api_config.json")
dry_run = False

# ensure output exists
os.makedirs(output_dir, exist_ok=True)


In [2]:
# Import basic modules
import pandas as pd
from datum_api_client import DatumApi
import datetime
from datetime import timedelta

# Import warnings
import warnings
warnings.filterwarnings("ignore")
# pip install xlrd
# pip install openpyxl

# Download excel with ticker




In [3]:
import os
from pathlib import Path

def find_orion_home() -> Path:
    # 1) найнадійніше — env var
    env = os.environ.get("ORION_HOME")
    if env:
        p = Path(env).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(f"ORION_HOME points to missing path: {p}")
        return p

    # 2) fallback: шукаємо папку OriON вгору від поточної директорії
    here = Path.cwd().resolve()
    for parent in [here] + list(here.parents):
        if parent.name.lower() == "orion":
            return parent
        cand = parent / "OriON"
        if cand.exists() and cand.is_dir():
            return cand.resolve()

    raise RuntimeError("Cannot locate OriON. Set ORION_HOME env var.")

ORION_HOME = find_orion_home()

CRACEN_DIR = ORION_HOME / "CRACEN"
WORK_DIR   = CRACEN_DIR / "work"     # батчі/маніфести/тимчасові файли
FINAL_PATH = CRACEN_DIR / "final.parquet"

CRACEN_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

BATCH_DIR = WORK_DIR / "CRACEN_batch"
BATCH_DIR.mkdir(parents=True, exist_ok=True)
ETF_DIR = WORK_DIR / "ETF"
ETF_DIR.mkdir(parents=True, exist_ok=True)


print("ORION_HOME:", ORION_HOME)
print("CRACEN_DIR :", CRACEN_DIR)
print("WORK_DIR  :", WORK_DIR)
print("FINAL_PATH:", FINAL_PATH)


ORION_HOME: C:\datum-api-examples-main\OriON
CRACEN_DIR : C:\datum-api-examples-main\OriON\CRACEN
WORK_DIR  : C:\datum-api-examples-main\OriON\CRACEN\work
FINAL_PATH: C:\datum-api-examples-main\OriON\CRACEN\final.parquet


In [4]:
from datetime import datetime, timedelta, date, time
from calendar import monthrange

# === ТЗ користувача: Europe/Kyiv ===
tz = None  # якщо нема zoneinfo, все одно працює локально

now_local = datetime.now(tz) if tz else datetime.now()
yesterday = (now_local - timedelta(days=1)).date()

# три місяці назад від учора (коректно для різної довжини місяців)
try:
    from dateutil.relativedelta import relativedelta
    start_date = (datetime.combine(yesterday, time(0, 0, 0)) - relativedelta(months=3)).date()
except Exception:
    # fallback без dateutil
    y, m, d = yesterday.year, yesterday.month - 3, yesterday.day
    while m <= 0:
        m += 12
        y -= 1
    d = min(d, monthrange(y, m)[1])
    start_date = date(y, m, d)

# формати з часом
start = f"{start_date:%Y-%m-%d} 00:00:00"
end   = f"{yesterday:%Y-%m-%d} 23:59:59"

# формати без часу
start_date_str = f"{start_date:%Y-%m-%d}"
end_date_str   = f"{yesterday:%Y-%m-%d}"

# вивід
print("🕓 Повні дати (з часом):")
print("start =", start)
print("end   =", end)
print("\n📅 Короткі дати (без часу):")
print("start_date_str =", start_date_str)
print("end_date_str   =", end_date_str)


🕓 Повні дати (з часом):
start = 2026-08-01 00:00:00
end   = 2026-11-01 23:59:59

📅 Короткі дати (без часу):
start_date_str = 2026-08-01
end_date_str   = 2026-11-01


In [5]:
import datetime

yesterday = (datetime.date.today() - datetime.timedelta(days=1)).strftime("%Y-%m-%d")

params = {
    "date_as_of": yesterday,
    "format": "json_records"
}

df = DatumApi.data_request("/calculations/median_premarket_value_traded_ex_finr_3m", params)

print(df.head())
print("Рядків:", len(df))


Signature has expired


  ticker      value
0      A   68669.72
1     AA  935733.26
2    AAA       0.00
3   AAAA       0.00
4   AAAB       0.00
Рядків: 13617


In [6]:
def filter_nonzero(df: pd.DataFrame) -> pd.DataFrame:
    """
    Повертає копію датафрейму лише з тими рядками,
    де колонка 'value' не дорівнює нулю.
    """
    if "value" not in df.columns:
        raise ValueError("У датафреймі немає колонки 'value'")
    return df[df["value"].fillna(0) != 0].copy()


# --- крок 1: фільтруємо df (де 'value' != 0)
df_filtered = filter_nonzero(df)

# --- крок 2: тягнемо дані по тікерах
ticker_params = {
    'fields': 'lvl3,market_cap',
    'active': True,
    'listed': True
}
tickers_df = DatumApi.data_request('/tickers', ticker_params)
# market_cap ніде далі не використовується (єдина згадка — у 'fields' вище),
# тому він не має відсіювати тикери: рядок викидаємо лише якщо бракує ticker або lvl3
tickers_df = tickers_df.dropna(subset=["ticker", "lvl3"])

# --- крок 3: додаємо потрібні колонки у df_filtered
TICKERS = df_filtered.merge(
    tickers_df,
    how='left',
    on='ticker'
)

print(f"Після фільтрації залишилось: {len(TICKERS):,} рядків")
TICKERS.head()


Після фільтрації залишилось: 7,098 рядків


,ticker,value,lvl3,market_cap
0,A,68669.72,Medical Equipment & Devices,49300.0
1,AA,935733.26,Metals & Mining,10342.0
2,AAAU,459786.34,NaN,NaN
3,AACG,26.70,Software,27.0
4,AADX,57173.51,Aerospace & Defense,1922.0


In [7]:
def get_unique_lvl2(df: pd.DataFrame) -> list:
    """
    Повертає список унікальних значень з колонки 'lvl3' у датафреймі.
    Пропуски (NaN) ігноруються.
    """
    if "lvl3" not in df.columns:
        raise ValueError("У датафреймі немає колонки 'lvl3'")
    
    unique_vals = sorted(df["lvl3"].dropna().unique().tolist())
    print(f"🔹 Знайдено {len(unique_vals)} унікальних значень у колонці 'lvl3'")
    return unique_vals


In [8]:
unique_lvl2 = get_unique_lvl2(TICKERS)
unique_lvl2


🔹 Знайдено 60 унікальних значень у колонці 'lvl3'


['Advertising & Marketing',
 'Aerospace & Defense',
 'Apparel & Textile Products',
 'Asset Management',
 'Automotive',
 'Banking',
 'Beverages',
 'Biotech & Pharma',
 'Cable & Satellite',
 'Chemicals',
 'Commercial Support Services',
 'Construction Materials',
 'Consumer Services',
 'Containers & Packaging',
 'Diversified Industrials',
 'E-Commerce Discretionary',
 'Electric Utilities',
 'Electrical Equipment',
 'Engineering & Construction',
 'Entertainment Content',
 'Food',
 'Forestry, Paper & Wood Products',
 'Gas & Water Utilities',
 'Health Care Facilities & Svcs',
 'Home & Office Products',
 'Home Construction',
 'Household Products',
 'IT Services',
 'Industrial Intermediate Prod',
 'Industrial Support Services',
 'Institutional Financial Svcs',
 'Insurance',
 'Internet Media & Services',
 'Investment Management',
 'Leisure Facilities & Services',
 'Leisure Products',
 'Machinery',
 'Medical Equipment & Devices',
 'Metals & Mining',
 'Oil & Gas Services & Equip',
 'Oil & Gas Sup

In [9]:
sector_to_etf = {
    # --- Finance ---
    "Asset Management": ["XLF", "IBIT"],
    "Banking": ["XLF", "KRE", "IBIT"],
    "Institutional Financial Svcs": ["XLF", "IBIT"],
    "Insurance": ["XLF"],
    "Specialty Finance": ["XLF", "IBIT"],

    # --- Tech / Internet / Growth ---
    "Software": ["IGV"],
    "IT Services": ["QQQ"],
    "Semiconductors": ["SOXX", "DRAM"],
    "Technology Hardware": ["QQQ", "SOXX"],
    "Internet Media & Services": ["QQQ", "KWEB", "FXI"],
    "E-Commerce Discretionary": ["QQQ", "KWEB", "XRT"],
    "Cable & Satellite": ["SPY", "IWM"],
    "Telecommunications": ["QQQ"],
    "Publishing & Broadcasting": ["QQQ"],
    "Entertainment Content": ["QQQ", "ARKK"],
    "Advertising & Marketing": ["QQQ"],

    # --- Industrials ---
    "Aerospace & Defense": ["NASA", "ITA"],
    "Diversified Industrials": ["XLB", "IWM"],
    "Industrial Intermediate Prod": ["XLB", "IWM"],
    "Industrial Support Services": ["IWM", "SPY"],
    "Engineering & Construction": ["XLB", "IWM"],
    "Machinery": ["XLB", "IWM"],
    "Transportation & Logistics": ["IWM", "SPY"],
    "Transportation Equipment": ["IWM", "SPY"],
    "Commercial Support Services": ["IWM", "SPY"],
    "Containers & Packaging": ["XLB", "IWM"],
    "Electrical Equipment": ["QQQ", "XLB"],

    # --- Energy / Materials / Commodities ---
    "Oil & Gas Services & Equip": ["XOP", "XLE", "UNG"],
    "Oil & Gas Supply Chain": ["XOP", "XLE", "UNG"],
    "Elec & Gas Marketing & Trading": ["XLU", "XLE"],
    "Electric Utilities": ["XLU"],
    "Gas & Water Utilities": ["XLU"],
    "Renewable Energy": ["XLE", "ARKK"],
    "Chemicals": ["XLB", "URA"],
    "Metals & Mining": ["GDX", "FCX", "COPX", "URA", "SLV"],
    "Steel": ["XLB", "GDX"],
    "Construction Materials": ["XLB", "IWM"],
    "Forestry, Paper & Wood Products": ["XLB", "IWM"],

    # --- Health Care ---
    "Biotech & Pharma": ["XBI"],
    "Medical Equipment & Devices": ["XLV"],
    "Health Care Facilities & Svcs": ["XLV", "IWM"],

    # --- Consumer Staples / Discretionary ---
    "Food": ["XLP"],
    "Beverages": ["XLP"],
    "Household Products": ["XLP"],
    "Tobacco & Cannabis": ["XLP"],
    "Retail - Consumer Staples": ["XLP", "XRT"],
    "Wholesale - Consumer Staples": ["XLP"],

    "Retail - Discretionary": ["XRT", "IWM", "SPY"],
    "Wholesale - Discretionary": ["XRT", "IWM"],
    "Consumer Services": ["IWM", "SPY"],
    "Leisure Facilities & Services": ["IWM", "SPY"],
    "Leisure Products": ["XRT", "IWM"],
    "Apparel & Textile Products": ["XRT", "IWM"],
    "Automotive": ["QQQ", "IWM"],
    "Home & Office Products": ["XRT", "IWM"],
    "Home Construction": ["XLB", "IWM"],

    # --- Real Estate ---
    "REIT": ["IWM", "SPY"],
    "Real Estate Owners & Developers": ["IWM", "SPY"],
    "Real Estate Services": ["IWM", "SPY"],
}

In [10]:
# ------------------------------------------------

In [11]:
import numpy as np
import pandas as pd


def calc_corr_beta_from_joined_gaps(df_pair: pd.DataFrame, x_col: str = "gap_x", y_col: str = "gap_y"):
    """
    Local corr/beta calculation without /calculations/corr_beta API.

    corr = Pearson(x, y)
    beta = cov(x, y) / var(y)
    """
    d = df_pair[[x_col, y_col]].copy()
    d[x_col] = pd.to_numeric(d[x_col], errors="coerce")
    d[y_col] = pd.to_numeric(d[y_col], errors="coerce")
    d = d.dropna()

    n = len(d)
    if n < 2:
        return np.nan, np.nan, n

    var_y = float(d[y_col].var(ddof=1))
    corr = float(d[x_col].corr(d[y_col]))

    if np.isnan(var_y) or var_y == 0:
        return corr, np.nan, n

    cov_xy = float(d[[x_col, y_col]].cov(ddof=1).iloc[0, 1])
    beta = cov_xy / var_y
    return corr, beta, n


def corr_beta_best_match_from_gaps(
    stock_gap_df: pd.DataFrame,
    etf_gap_df: pd.DataFrame,
    x_tickers,
    y_tickers,
    *,
    min_overlap: int = 20,
    output_path: str = None,
):
    """
    Pick best Y per X using local gaps.

    stock_gap_df: ['ticker','date','gap']
    etf_gap_df:   ['ticker','date','gap']
    """
    sx = stock_gap_df.copy()
    sy = etf_gap_df.copy()

    for df in (sx, sy):
        if not {"ticker", "date", "gap"}.issubset(df.columns):
            raise KeyError("Expected columns ['ticker','date','gap'] in gap data")
        df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()
        df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.strftime("%Y-%m-%d")
        df["gap"] = pd.to_numeric(df["gap"], errors="coerce")

    s_map = {t: g[["date", "gap"]].copy() for t, g in sx.groupby("ticker")}
    e_map = {t: g[["date", "gap"]].copy() for t, g in sy.groupby("ticker")}

    rows_all = []
    rows_best = []

    for x in [str(v).upper() for v in x_tickers]:
        g_x = s_map.get(x)
        if g_x is None or g_x.empty:
            continue

        best = None
        for y in [str(v).upper() for v in y_tickers]:
            g_y = e_map.get(y)
            if g_y is None or g_y.empty:
                continue

            m = g_x.merge(g_y, on="date", how="inner", suffixes=("_x", "_y")).dropna()
            corr, beta, n = calc_corr_beta_from_joined_gaps(m, x_col="gap_x", y_col="gap_y")
            if n < int(min_overlap) or pd.isna(corr):
                continue

            row = {
                "x_ticker": x,
                "y_ticker": y,
                "corr": corr,
                "beta": beta,
                "n_overlap": n,
            }
            rows_all.append(row)
            if best is None or row["corr"] > best["corr"]:
                best = row

        if best is not None:
            rows_best.append({
                "x_ticker": best["x_ticker"],
                "best_y_ticker": best["y_ticker"],
                "best_corr": best["corr"],
                "beta_with_best": best["beta"],
                "n_overlap": best["n_overlap"],
            })

    df_all = pd.DataFrame(rows_all)
    df_pairs = pd.DataFrame(rows_best)

    if df_pairs.empty:
        df_pairs = pd.DataFrame(columns=["x_ticker", "best_y_ticker", "best_corr", "beta_with_best", "n_overlap"])

    if output_path:
        df_pairs.to_csv(output_path, index=False)

    return df_all, df_pairs



In [12]:
# Кеш сирого фетчу. /daily/gaps і /reports тягнулись ДВІЧІ за прогін — тут і в
# комірці 42 — тобто ~14 500 зайвих запитів, і саме вони й провокували 429.
# Складаємо результат ДО відкидання викидів: комірка 42 рахує sigma на СИРИХ
# гепах, і підстановка вже обрізаних занизила б її.
CRACEN_FETCH_CACHE = {}


def corr_beta_best_etf_by_lvl3(
    tickers_df,
    sector_to_etf,
    sector_col="lvl3",
    ticker_col="ticker",
    output_path="CRACEN/work/x_to_best_etf_by_lvl3.csv",
    bench_path="ticker_bench.csv",
    fallback_etfs=None,
    start_date=None,
    end_date=None,
    last_n_reports=20,
    cap_q=0.8,
    min_hist_for_cap=20,
    min_overlap=20,
    max_workers_net=8,        # 64 потоки без пауз давали суцільні 429
    max_fail_ratio=0.15,      # 4.4% остаточних відмов виміряно як НОРМУ для цього
                              # API навіть з retry; бар упритул до норми валив би
                              # прогін через раз. Ловить справжню відмову, не шум.
):
    import random
    import time
    import numpy as np
    import pandas as pd
    from concurrent.futures import ThreadPoolExecutor

    if fallback_etfs is None:
        fallback_etfs = ["SPY", "QQQ", "IWM"]
    if start_date is None:
        start_date = globals().get("start_date_str")
    if end_date is None:
        end_date = globals().get("end_date_str")
    if not start_date or not end_date:
        raise ValueError("Provide start_date/end_date or define start_date_str/end_date_str")
    if sector_col not in tickers_df.columns:
        raise ValueError(f"tickers_df has no '{sector_col}' column")
    if ticker_col not in tickers_df.columns:
        raise ValueError(f"tickers_df has no '{ticker_col}' column")

    s_date, e_date = str(start_date)[:10], str(end_date)[:10]

    def _pick_col(df, candidates):
        cols_l = {c.lower().strip(): c for c in df.columns}
        return next((cols_l[c.lower()] for c in candidates if c.lower() in cols_l), None)

    _fails = {"gaps": 0, "reports": 0}

    def _request_with_retry(endpoint, params, *, max_retries=6, backoff=0.5):
        """Повторює запит при 429/мережевих збоях з наростаючою паузою.
        Той самий підхід, що в /intraday/v3. Вичерпавши спроби — кидає
        останній виняток, який ловить виклик вище."""
        for attempt in range(max_retries + 1):
            try:
                return DatumApi.data_request(endpoint, params)
            except Exception:
                if attempt == max_retries:
                    raise
                time.sleep(backoff * (2 ** attempt) * (0.9 + 0.2 * random.random()))

    def _fetch_gap(t):
        try:
            dfg = _request_with_retry("/daily/gaps", {
                "ticker": t, "start_date": s_date, "end_date": e_date, "format": "json_records",
            })
            if dfg is None or dfg.empty:
                return None
            dcol = _pick_col(dfg, ["date", "move_date", "dt", "datetime", "day"])
            if dcol is None or "gap" not in dfg.columns:
                return None
            out = pd.DataFrame({
                "ticker": t,
                "date": pd.to_datetime(dfg[dcol], errors="coerce").dt.strftime("%Y-%m-%d"),
                "gap": pd.to_numeric(dfg["gap"], errors="coerce"),
            }).dropna()
            return out if not out.empty else None
        except Exception as ex:
            _fails["gaps"] += 1
            print(f"WARN gaps fail {t}: {ex}")
            return None

    def _fetch_report(t):
        try:
            rep = _request_with_retry("/reports", {
                "ticker": t, "start_move_date": s_date, "end_move_date": e_date,
            })
            if rep is None or rep.empty:
                return None
            dcol = _pick_col(rep, ["report_date", "move_date", "date", "dt", "datetime", "published_at", "created_at"])
            if dcol is None:
                return None
            out = pd.DataFrame({
                "ticker": t,
                "date": pd.to_datetime(rep[dcol], errors="coerce").dt.strftime("%Y-%m-%d"),
            }).dropna().sort_values("date").tail(int(last_n_reports))
            return out if not out.empty else None
        except Exception as ex:
            _fails["reports"] += 1
            print(f"WARN reports fail {t}: {ex}")
            return None

    # ── prepare ──────────────────────────────────────────────────────────────
    df_clean = tickers_df[[ticker_col, sector_col]].drop_duplicates().copy()
    df_clean[ticker_col] = df_clean[ticker_col].astype(str).str.strip().str.upper()
    df_clean[sector_col] = df_clean[sector_col].fillna("UNKNOWN").astype(str)

    stocks = sorted(df_clean[ticker_col].dropna().unique())
    etf_candidates = set(fallback_etfs)
    for sec in df_clean[sector_col].unique():
        etf_candidates.update(sector_to_etf.get(sec, fallback_etfs) or fallback_etfs)
    etf_candidates = sorted(str(x).upper() for x in etf_candidates)

    print(f"START gaps: stocks={len(stocks)} etfs={len(etf_candidates)} window={s_date}..{e_date}")

    # ── parallel fetch: all gaps + reports in one pool ────────────────────────
    all_gap_tickers = sorted(set(stocks) | set(etf_candidates))
    with ThreadPoolExecutor(max_workers=min(max_workers_net,
                                            len(all_gap_tickers) + len(stocks))) as ex:
        gap_futs  = {t: ex.submit(_fetch_gap,    t) for t in all_gap_tickers}
        rep_futs  = {t: ex.submit(_fetch_report, t) for t in stocks}

    gap_res = {t: f.result() for t, f in gap_futs.items()}
    rep_res = {t: f.result() for t, f in rep_futs.items()}

    _total = len(gap_futs) + len(rep_futs)
    _failed = _fails["gaps"] + _fails["reports"]
    if _total and _failed / _total > max_fail_ratio:
        raise RuntimeError(
            f"API відмовив на {_failed} з {_total} запитів "
            f"({_failed / _total * 100:.1f}% > {max_fail_ratio * 100:.0f}%). "
            "Далі вийшла б порожня або обрізана мапа bench/corr/beta."
        )

    def _concat(parts, cols):
        valid = [p for p in parts if p is not None]
        return pd.concat(valid, ignore_index=True) if valid else pd.DataFrame(columns=cols)

    stock_gaps = _concat([gap_res.get(t) for t in stocks],         ["ticker", "date", "gap"])
    etf_gaps   = _concat([gap_res.get(t) for t in etf_candidates], ["ticker", "date", "gap"])
    reports    = _concat([rep_res.get(t) for t in stocks],          ["ticker", "date"]).drop_duplicates()

    CRACEN_FETCH_CACHE.clear()
    CRACEN_FETCH_CACHE.update({
        "window": (s_date, e_date),
        "last_n_reports": int(last_n_reports),
        "tickers": set(stocks),
        "gaps": stock_gaps.copy(),      # сирі, до drop anomalies нижче
        "reports": reports.copy(),
    })

    # ── exclude report days ──────────────────────────────────────────────────
    if not reports.empty and not stock_gaps.empty:
        m = stock_gaps.merge(reports.assign(_rep=1), on=["ticker", "date"], how="left")
        stock_gaps = m[m["_rep"].isna()][["ticker", "date", "gap"]].copy()

    # ── drop anomalies (vectorised) ──────────────────────────────────────────
    if not stock_gaps.empty:
        stock_gaps["gap"] = pd.to_numeric(stock_gaps["gap"], errors="coerce")
        abs_gap = stock_gaps["gap"].abs()
        cnts = stock_gaps.groupby("ticker")["gap"].transform("count")
        eligible = cnts >= min_hist_for_cap
        if eligible.any():
            caps = (stock_gaps[eligible]
                    .groupby("ticker")["gap"]
                    .agg(lambda x: np.quantile(x.abs(), cap_q)))
            cap_mapped = stock_gaps["ticker"].map(caps)
            stock_gaps = stock_gaps[(~eligible) | (abs_gap <= cap_mapped)].reset_index(drop=True)

    # ── build ETF numpy matrix: rows=dates, cols=etfs ───────────────────────
    if not etf_gaps.empty:
        etf_pivot = etf_gaps.pivot_table(index="date", columns="ticker", values="gap", aggfunc="first")
        date_to_row = {d: i for i, d in enumerate(etf_pivot.index)}
        etf_to_col  = {c: i for i, c in enumerate(etf_pivot.columns)}
        etf_arr     = etf_pivot.to_numpy(dtype=float)
    else:
        date_to_row = {}; etf_to_col = {}; etf_arr = np.empty((0, 0))

    stock_map = (
        {t: g.set_index("date")["gap"] for t, g in stock_gaps.groupby("ticker")}
        if not stock_gaps.empty else {}
    )

    # ── corr/beta loop ───────────────────────────────────────────────────────
    rows_all = []
    rows_best = []
    sectors_with_fallback = []

    for sec in sorted(df_clean[sector_col].unique()):
        etfs_raw = sector_to_etf.get(sec)
        if not etfs_raw:
            etfs_raw = list(fallback_etfs)
            sectors_with_fallback.append(sec)
        etfs = [str(e).upper() for e in etfs_raw]
        etfs_ok  = [e for e in etfs if e in etf_to_col]
        col_idxs = [etf_to_col[e] for e in etfs_ok]
        if not col_idxs:
            continue

        for st in sorted(df_clean.loc[df_clean[sector_col] == sec, ticker_col].unique()):
            sx = stock_map.get(st)
            if sx is None or sx.empty:
                continue

            st_dates = [d for d in sx.index if d in date_to_row]
            if not st_dates:
                continue

            row_idxs = [date_to_row[d] for d in st_dates]
            x_vals   = sx[st_dates].to_numpy(dtype=float)
            y_mat    = etf_arr[np.ix_(row_idxs, col_idxs)]   # (n_dates, n_etfs)
            valid_x  = ~np.isnan(x_vals)

            best_corr = -np.inf
            best_etf  = None
            best_beta = np.nan
            best_n    = 0

            for j, etf in enumerate(etfs_ok):
                mask = valid_x & ~np.isnan(y_mat[:, j])
                n = int(mask.sum())
                if n < min_overlap:
                    continue

                xv = x_vals[mask];  yv = y_mat[mask, j]
                xc = xv - xv.mean(); yc = yv - yv.mean()
                var_y = float((yc * yc).sum() / (n - 1))
                if var_y == 0 or np.isnan(var_y):
                    continue
                cov_xy = float((xc * yc).sum() / (n - 1))
                std_x  = float(np.sqrt((xc * xc).sum() / (n - 1)))
                if std_x == 0:
                    continue
                corr = cov_xy / (std_x * np.sqrt(var_y))
                beta = cov_xy / var_y

                rows_all.append({"x_ticker": st, "y_ticker": etf, "corr": corr, "beta": beta, "lvl3": sec, "n_overlap": n})

                if corr > best_corr:
                    best_corr, best_etf, best_beta, best_n = corr, etf, beta, n

            if best_etf is not None:
                rows_best.append({"x_ticker": st, "best_y_ticker": best_etf, "best_corr": best_corr,
                                  "beta_with_best": best_beta, "lvl3": sec, "n_overlap": best_n})

    # ── output ───────────────────────────────────────────────────────────────
    df_all   = pd.DataFrame(rows_all)
    df_pairs = pd.DataFrame(rows_best) if rows_best else pd.DataFrame(
        columns=["x_ticker", "best_y_ticker", "best_corr", "beta_with_best", "lvl3", "n_overlap"])

    df_pairs.to_csv(output_path, index=False)
    df_bench = df_pairs.rename(columns={"x_ticker": "ticker", "best_y_ticker": "benchmark"})[["ticker", "benchmark"]]
    df_bench.to_csv(bench_path, index=False)

    print(f"DONE pairs: {len(df_pairs)} -> {output_path}")
    print(f"DONE bench map: {len(df_bench)} -> {bench_path}")
    if sectors_with_fallback:
        print("Fallback ETF used for sectors:", ", ".join(map(str, sectors_with_fallback[:20])))

    return df_all, df_pairs


In [13]:
# Local corr/beta: stock gaps vs ETF gaps, excluding report days and stock outliers
df_all, df_pairs = corr_beta_best_etf_by_lvl3(
    tickers_df=TICKERS,
    sector_to_etf=sector_to_etf,
    sector_col="lvl3",
    ticker_col="ticker",
    output_path=str(WORK_DIR / "x_to_best_etf_by_lvl3.csv"),
    bench_path=str(WORK_DIR / "ticker_bench_.csv"),
    start_date=start_date_str,
    end_date=end_date_str,
    last_n_reports=20,
    cap_q=0.8,
    min_hist_for_cap=20,
    min_overlap=20,
)

# Prepare parquet for downstream merge (cell 31)
corr_beta_pairs = df_pairs.rename(columns={
    "best_y_ticker": "y_ticker",
    "best_corr": "corr",
    "beta_with_best": "beta",
})[["x_ticker", "y_ticker", "corr", "beta"]].copy()

corr_beta_pairs.to_parquet(str(WORK_DIR / "corr_beta_pairs.parquet"), index=False)

print(df_all.head())
print("Rows in raw stock-etf pairs:", len(df_all))
print(df_pairs.head())
print("Saved:", str(WORK_DIR / "corr_beta_pairs.parquet"))



START gaps: stocks=7098 etfs=28 window=2026-08-01..2026-11-01

DONE pairs: 7006 -> C:\datum-api-examples-main\OriON\CRACEN\work\x_to_best_etf_by_lvl3.csv
DONE bench map: 7006 -> C:\datum-api-examples-main\OriON\CRACEN\work\ticker_bench_.csv
Fallback ETF used for sectors: Investment Management, Other Financial Services, UNKNOWN
  x_ticker y_ticker      corr      beta                     lvl3  n_overlap
0      ADV      QQQ  0.025439  0.018256  Advertising & Marketing         50
1     AEHL      QQQ -0.057463 -0.236992  Advertising & Marketing         51
2      APP      QQQ  0.558626  0.558849  Advertising & Marketing         50
3     BAOS      QQQ  0.077364  0.204684  Advertising & Marketing         51
4     CDLX      QQQ  0.241932  0.277231  Advertising & Marketing         50
Rows in raw stock-etf pairs: 15757
  x_ticker best_y_ticker  best_corr  beta_with_best                     lvl3  \
0      ADV           QQQ   0.025439        0.018256  Advertising & Marketing   
1     AEHL           QQQ  -0.057463       -0.236992  Advertising & Marketing   
2  

In [14]:
# ================== FAST INTRADAY SUITE w/ BLUE OCEAN (updated + rolling prune) ==================
import os, gc, time, glob, re, csv, uuid, random, threading
import pandas as pd
from datetime import timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Optional, Tuple

# ---------- utils ----------
def _p(*args, **kwargs):
    kwargs.setdefault("flush", True)
    print(*args, **kwargs)

_PSUTIL = False
try:
    import psutil
    _PSUTIL = True
except Exception:
    psutil = None

def _proc_rss_mb() -> float:
    if _PSUTIL:
        try:
            return psutil.Process(os.getpid()).memory_info().rss / (1024**2)
        except Exception:
            pass
    try:
        import resource
        ru = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        if ru > 10**9:
            return ru / (1024**2)
        return ru / 1024.0
    except Exception:
        pass
    try:
        import tracemalloc
        if not tracemalloc.is_tracing():
            tracemalloc.start()
        cur, _ = tracemalloc.get_traced_memory()
        return cur / (1024**2)
    except Exception:
        return float('nan')

def _mem_str(prefix: str = "") -> str:
    mb = _proc_rss_mb()
    if mb != mb:
        return f"{prefix}🧠 RAM: n/a"
    return f"{prefix}🧠 RAM: {mb:,.1f} MB"

def _ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)

def _fmt_eta(sec: float) -> str:
    sec = max(0, int(sec))
    d, r = divmod(sec, 86400)
    h, r = divmod(r, 3600)
    m, s = divmod(r, 60)
    if d > 0: return f"{d}d {h:02d}:{m:02d}:{s:02d}"
    if h > 0: return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

def _next_batch_index(out_dir: str, out_prefix: str) -> int:
    pat_csv = os.path.join(out_dir, f"{out_prefix}_*.csv")
    pat_gz  = os.path.join(out_dir, f"{out_prefix}_*.csv.gz")
    files = glob.glob(pat_csv) + glob.glob(pat_gz)
    mx = 0
    rx = re.compile(rf'^{re.escape(out_prefix)}_(\d+)\.csv(?:\.gz)?$')
    for f in files:
        m = rx.match(os.path.basename(f))
        if m:
            mx = max(mx, int(m.group(1)))
    return mx + 1 if mx > 0 else 1

# ---------- збої API != порожні дані ----------
# Раніше будь-який збій (429/5xx/мережа; DatumApi.data_request повертає None після
# вичерпання ретраїв) тихо перетворювався на порожній DataFrame, а порожній результат
# позначав ВСІ дні діапазону «готовими» у done_days.csv → ці дні більше ніколи не
# докачувались (NVDA/AAPL/INTC… без 10–28 серпня при «готових» днях у маніфесті).
# Тепер збій піднімає FetchFailed, і такі дні НЕ потрапляють у маніфест.
class FetchFailed(Exception):
    pass

_MANIFEST_LOCK = threading.Lock()   # done_days.csv пишуть і воркери, і _flush → без lock рядки рвались

def _data_request_checked(endpoint: str, params: dict, max_retries: int, retry_backoff: float) -> pd.DataFrame:
    """DataFrame (можливо порожній = API відповів 200 і барів справді нема) або FetchFailed."""
    attempts = max(0, int(max_retries)) + 1
    last = None
    for attempt in range(attempts):
        try:
            df = DatumApi.data_request(endpoint, params)
            if isinstance(df, pd.DataFrame):
                return df
            last = "None (DatumApi вичерпав ретраї)"
        except Exception as e:
            code = getattr(getattr(e, "response", None), "status_code", None)
            if code in (400, 404, 422):      # запит/тікер не обслуговується — не тимчасовий збій
                return pd.DataFrame()
            last = f"{type(e).__name__}: {e}"
        if attempt < attempts - 1:
            time.sleep(retry_backoff * (2 ** attempt) * (0.9 + 0.2 * random.random()))
    raise FetchFailed(f"{endpoint} {params.get('ticker')} {params.get('start_datetime')}..{params.get('end_datetime')}: {last}")

# ---------- manifest helpers (old ticker-level) ----------
def _manifest_path(out_dir: str) -> str:
    return os.path.join(out_dir, "done_tickers.txt")

def _load_done_tickers_manifest(out_dir: str) -> Optional[set]:
    path = _manifest_path(out_dir)
    if not os.path.exists(path):
        return None
    done = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            t = line.strip()
            if t:
                done.add(t)
    return done

def _append_done_tickers_manifest(out_dir: str, tickers: List[str]):
    if not tickers:
        return
    path = _manifest_path(out_dir)
    existing = set()
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                s = line.strip()
                if s:
                    existing.add(s)
    new_ones = [t for t in set(tickers) if t not in existing]
    if not new_ones:
        return
    with open(path, "a", encoding="utf-8") as f:
        for t in new_ones:
            f.write(t + "\n")

# ---------- day-manifest helpers (NEW) ----------
def _manifest_days_path(out_dir: str) -> str:
    return os.path.join(out_dir, "done_days.csv")

def _load_done_days_manifest(out_dir: str) -> dict:
    """
    Return dict: ticker -> set({'YYYY-MM-DD', ...})
    """
    path = _manifest_days_path(out_dir)
    if not os.path.exists(path):
        return {}
    done = {}
    with open(path, "r", encoding="utf-8", newline="") as f:
        rdr = csv.reader(f)
        header = next(rdr, None)
        col_idx = {"ticker": 0, "date": 1}
        if header and "ticker" in header and "date" in header:
            col_idx["ticker"] = header.index("ticker")
            col_idx["date"]   = header.index("date")
        else:
            f.seek(0); rdr = csv.reader(f)
        for row in rdr:
            if not row:
                continue
            tk = row[col_idx["ticker"]].strip()
            dt = row[col_idx["date"]].strip()
            if tk and dt:
                done.setdefault(tk, set()).add(dt)
    return done

def _append_done_days_manifest(out_dir: str, df: pd.DataFrame):
    """
    From df uses 'ticker' and 'dt' (YYYY-MM-DD HH:MM:SS) to append (ticker,date) to done_days.csv
    """
    if df is None or df.empty or "dt" not in df.columns or "ticker" not in df.columns:
        return
    path = _manifest_days_path(out_dir)
    _ensure_dir(out_dir)

    tmp = df[["ticker","dt"]].copy()
    tmp["date"] = tmp["dt"].astype(str).str.slice(0,10)
    tmp = tmp.drop_duplicates(subset=["ticker","date"])

    with _MANIFEST_LOCK:
        need_header = not os.path.exists(path)
        with open(path, "a", encoding="utf-8", newline="") as f:
            wr = csv.writer(f)
            if need_header:
                wr.writerow(["ticker","date"])
            for tk, dd in zip(tmp["ticker"], tmp["date"]):
                wr.writerow([str(tk).strip(), str(dd).strip()])

def prune_done_days_manifest(out_dir: str, start_date_str: str, end_date_str: str) -> dict:
    """
    Rolling-window prune for done_days.csv:
    - keep only rows where start_date_str <= date <= end_date_str
    - returns refreshed dict: ticker -> set(dates)
    Safe to call even if file doesn't exist.
    """
    path = _manifest_days_path(out_dir)
    if not os.path.exists(path):
        return {}

    # normalize inputs
    start_date_str = str(start_date_str)[:10]
    end_date_str   = str(end_date_str)[:10]
    if start_date_str > end_date_str:
        # swap just in case
        start_date_str, end_date_str = end_date_str, start_date_str

    rows = []
    with open(path, "r", encoding="utf-8", newline="") as f:
        rdr = csv.reader(f)
        header = next(rdr, None)
        col_idx = {"ticker": 0, "date": 1}
        has_header = bool(header and "ticker" in header and "date" in header)
        if has_header:
            col_idx["ticker"] = header.index("ticker")
            col_idx["date"]   = header.index("date")
        else:
            # no header; treat first row as data
            if header:
                rdr = [header] + list(rdr)  # include first row back as data
            else:
                rdr = []
        for row in rdr:
            if not row:
                continue
            try:
                tk = str(row[col_idx["ticker"]]).strip().upper()
                dt = str(row[col_idx["date"]]).strip()[:10]
            except Exception:
                continue
            if not tk or not dt:
                continue
            if start_date_str <= dt <= end_date_str:
                rows.append((tk, dt))

    # rewrite file (dedup)
    _ensure_dir(out_dir)
    tmp_path = path + ".tmp"
    seen = set()
    with open(tmp_path, "w", encoding="utf-8", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(["ticker","date"])
        for tk, dt in rows:
            key = (tk, dt)
            if key in seen:
                continue
            seen.add(key)
            wr.writerow([tk, dt])
    os.replace(tmp_path, path)

    # build dict cache
    done = {}
    for tk, dt in seen:
        done.setdefault(tk, set()).add(dt)
    return done

def _dates_between(start: str, end: str) -> List[pd.Timestamp]:
    s = pd.to_datetime(start)
    e = pd.to_datetime(end)
    if pd.isna(s) or pd.isna(e) or s > e:
        return []
    days = pd.date_range(start=s.normalize(), end=e.normalize(), freq="D")
    return list(days)

def _group_consecutive_dates(dates: List[pd.Timestamp], max_span_days: int) -> List[Tuple[str, str]]:
    """
    Build [start_dt, end_dt] ranges from a list of dates, merging consecutive days,
    capped by max_span_days per range. Datetimes are ISO to seconds.
    """
    if not dates:
        return []
    dates = sorted(pd.to_datetime(dates))
    out = []
    run_start = dates[0]
    run_end = dates[0]
    for d in dates[1:]:
        consecutive = (d - run_end).days == 1
        span_len = (d - run_start).days + 1
        if consecutive and span_len <= max_span_days:
            run_end = d
        else:
            a = run_start.strftime("%Y-%m-%d 00:00:00")
            b = (run_end + pd.Timedelta(hours=23, minutes=59, seconds=59)).strftime("%Y-%m-%d %H:%M:%S")
            out.append((a, b))
            run_start = d
            run_end = d
    a = run_start.strftime("%Y-%m-%d 00:00:00")
    b = (run_end + pd.Timedelta(hours=23, minutes=59, seconds=59)).strftime("%Y-%m-%d %H:%M:%S")
    out.append((a, b))
    return out

# ================== SMALL (low-level) FUNCTIONS ==================
def _build_ranges(start: str, end: str, chunk_days: int) -> List[Tuple[str, str]]:
    """Розбиває [start,end] на неперекривні відрізки по chunk_days днів."""
    chunk_days = max(1, int(chunk_days))
    s = pd.to_datetime(start)
    e = pd.to_datetime(end)
    if pd.isna(s) or pd.isna(e) or s > e:
        return []
    out: List[Tuple[str, str]] = []
    cur = s
    step = timedelta(days=chunk_days) - timedelta(seconds=1)
    while cur <= e:
        r_end = min(cur + step, e)
        out.append((cur.strftime("%Y-%m-%d %H:%M:%S"), r_end.strftime("%Y-%m-%d %H:%M:%S")))
        cur = r_end + timedelta(seconds=1)
    return out

def _one_call_v3_dt_fast(
    ticker: str,
    start_dt: str,
    end_dt: str,
    interval: int,
    *,
    chart_type: str = "ohlcv",
    include_boats: bool = False,
    max_retries: int = 4,
    retry_backoff: float = 0.5,     # сек
    normalize_ohlcv: bool = False,  # дефолт: ВИМКНЕНО (швидкість)
) -> pd.DataFrame:
    """Основний ендпоінт: /intraday/v3. Збій → FetchFailed (не порожній DF)."""
    params = {
        "ticker": ticker.upper(),
        "interval": int(interval),
        "chart_type": chart_type,
        "start_datetime": start_dt,
        "end_datetime": end_dt,
        "include_boats": bool(include_boats),
        "format": "json_records",
    }
    df = _data_request_checked("/intraday/v3", params, max_retries, retry_backoff)
    if df.empty:
        return pd.DataFrame()
    if "ticker" not in df.columns:
        df.insert(0, "ticker", ticker.upper())
    if normalize_ohlcv:
        for col in ("o","h","l","c","v"):
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="ignore")
    return df

def _one_call_blue_ocean_v3_dt_fast(
    ticker: str,
    start_dt: str,
    end_dt: str,
    interval: int,
    *,
    chart_type: str = "ohlcv",
    max_retries: int = 4,
    retry_backoff: float = 0.5,
    normalize_ohlcv: bool = False,
) -> pd.DataFrame:
    """Blue Ocean ендпоінт: /intraday/v3/blue_ocean (20:00–04:00). Збій → FetchFailed."""
    params = {
        "ticker": ticker.upper(),
        "interval": int(interval),
        "chart_type": chart_type,
        "start_datetime": start_dt,
        "end_datetime": end_dt,
        "format": "json_records",
    }
    df = _data_request_checked("/intraday/v3/blue_ocean", params, max_retries, retry_backoff)
    if df.empty:
        return pd.DataFrame()
    if "ticker" not in df.columns:
        df.insert(0, "ticker", ticker.upper())
    if normalize_ohlcv:
        for col in ("o","h","l","c","v"):
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="ignore")
    return df

def fetch_intraday_v3_datetime_fast(
    ticker: str,
    start: str,
    end: str,
    interval: int = 1,
    *,
    chunk_days: int = 14,
    parallel: bool = True,
    max_workers: int = 12,
    chart_type: str = "ohlcv",
    include_boats: bool = False,
    sort_by_dt: bool = False,   # дефолт: викл (швидкість)
    dedup_dt: bool = False,     # дефолт: викл (швидкість)
    normalize_ohlcv: bool = False,  # дефолт: викл (швидкість)
    include_blue_ocean: bool = False,   # ⬅️ НОВЕ
) -> pd.DataFrame:
    """
    Тягне інтрадійні дані основного вікна і, опційно, Blue Ocean (20:00–04:00),
    зшиває в один DF. За замовчуванням без зайвих перетворень.
    """
    ranges = _build_ranges(start, end, chunk_days)
    if not ranges:
        return pd.DataFrame()

    worker_count = max(1, min(int(max_workers), len(ranges) * (2 if include_blue_ocean else 1))) if parallel else 1
    parts: List[pd.DataFrame] = []

    if parallel and (len(ranges) > 1 or include_blue_ocean):
        with ThreadPoolExecutor(max_workers=worker_count) as ex:
            futs = []
            for (a, b) in ranges:
                futs.append(ex.submit(
                    _one_call_v3_dt_fast, ticker, a, b, interval,
                    chart_type=chart_type, include_boats=include_boats,
                    normalize_ohlcv=normalize_ohlcv
                ))
                if include_blue_ocean:
                    futs.append(ex.submit(
                        _one_call_blue_ocean_v3_dt_fast, ticker, a, b, interval,
                        chart_type=chart_type, normalize_ohlcv=normalize_ohlcv
                    ))
            for f in as_completed(futs):
                df = f.result()
                if df is not None and not df.empty:
                    parts.append(df)
    else:
        for (a, b) in ranges:
            df_main = _one_call_v3_dt_fast(
                ticker, a, b, interval,
                chart_type=chart_type, include_boats=include_boats,
                normalize_ohlcv=normalize_ohlcv
            )
            if df_main is not None and not df_main.empty:
                parts.append(df_main)
            if include_blue_ocean:
                df_bo = _one_call_blue_ocean_v3_dt_fast(
                    ticker, a, b, interval,
                    chart_type=chart_type, normalize_ohlcv=normalize_ohlcv
                )
                if df_bo is not None and not df_bo.empty:
                    parts.append(df_bo)

    if not parts:
        return pd.DataFrame()

    out = pd.concat(parts, ignore_index=True)

    # Якщо підмішували Blue Ocean — відсортуємо по 'dt'
    if "dt" in out.columns and (include_blue_ocean or sort_by_dt or dedup_dt):
        if dedup_dt:
            out = out.drop_duplicates(subset=["ticker", "dt"], keep="last")
        out = out.sort_values("dt", kind="mergesort").reset_index(drop=True)

    return out

# ================== BIG (all tickers, batch writer) [UPDATED + rolling prune] ==================
def fetch_intraday_v3_for_all_tickers(
    tickers_df: pd.DataFrame,
    start: str,
    end: str,
    *,
    interval: int = 1,
    chunk_days: int = 30,
    parallel_chunks: bool = True,
    max_workers_chunks: int = 12,
    parallel_tickers: bool = True,
    max_workers_tickers: int = 6,
    save_every: int = 50,
    out_dir: str = "intraday_batches",
    out_prefix: str = "batch",
    gzip: bool = True,
    keep_columns: Optional[List[str]] = None,
    verbose: bool = True,
    progress_every: int = 10,
    mem_log_every: int = 50,
    use_manifest: bool = False,            # old ticker-level manifest (off by default)
    use_day_manifest: bool = True,         # NEW: day-level manifest (on by default)
    flush_when_rows: Optional[int] = None,
    flush_every_seconds: Optional[int] = None,
    include_blue_ocean: bool = False,      # pass-through
) -> pd.DataFrame:
    assert "ticker" in tickers_df.columns, "tickers_df має містити колонку 'ticker'"

    # ⬆️ НОРМАЛІЗАЦІЯ: тікери у верхній регістр одразу на вході
    tickers: List[str] = [str(t).strip().upper() for t in tickers_df["ticker"].dropna().unique()]
    if not tickers:
        return pd.DataFrame(columns=["ticker","error"])

    _ensure_dir(out_dir)

    # --- rolling window bounds as YYYY-MM-DD for manifest prune ---
    start_date_str = str(start)[:10]
    end_date_str   = str(end)[:10]

    has_batches = bool(glob.glob(os.path.join(out_dir, f"{out_prefix}_*.csv*")))
    done_tickers = set()
    if use_manifest:
        m = _load_done_tickers_manifest(out_dir)
        if m and has_batches:
            done_tickers = m
        elif m and not has_batches and verbose:
            _p("⚠️ Знайшов done_tickers.txt, але немає batch-файлів → ігнорую маніфест.")
        # apply old manifest only if explicitly enabled
        tickers = [t for t in tickers if t not in done_tickers]

    # NEW: day manifest load (+ rolling prune)
    if use_day_manifest:
        # prune done_days.csv to the current rolling window and get fresh dict
        done_days_by_ticker = prune_done_days_manifest(out_dir, start_date_str, end_date_str)
    else:
        done_days_by_ticker = {}

    # ⬆️ НОРМАЛІЗАЦІЯ: ключі маніфесту у верхній регістр
    if use_day_manifest and done_days_by_ticker:
        done_days_by_ticker = {str(k).upper(): set(v) for k, v in done_days_by_ticker.items()}

    # --- попереднє відсікання повністю готових тікерів ---
    all_days_list = _dates_between(start, end)
    all_days_str = [d.strftime("%Y-%m-%d") for d in all_days_list]

    def _ticker_fully_done(tk: str) -> bool:
        if not use_day_manifest:
            return False
        s = done_days_by_ticker.get(str(tk).upper(), set())
        return bool(s) and all(dd in s for dd in all_days_str)

    fully_done = [t for t in tickers if _ticker_fully_done(t)]
    tickers = [t for t in tickers if t not in fully_done]

    total = len(tickers)
    batch_idx = _next_batch_index(out_dir, out_prefix)
    run_id = str(uuid.uuid4())[:8]

    errors, buffer = [], []
    written_total = 0
    start_time = time.time()
    last_flush_at = start_time
    rows_in_buffer = 0

    if verbose:
        psutil_note = "psutil" if _PSUTIL else "fallback"

        # логіка визначення діапазонів дат для логів
        if use_day_manifest and done_days_by_ticker:
            # усі дні в маніфесті
            all_done_dates = sorted({d for s in done_days_by_ticker.values() for d in s})
            skipped_days_str = (
                f"{all_done_dates[0]} → {all_done_dates[-1]} ({len(all_done_dates)} днів)"
                if all_done_dates else "–"
            )
            # усі дні, які ще не оброблені (порівняно з повним діапазоном)
            all_days_full = [d.strftime("%Y-%m-%d") for d in _dates_between(start, end)]
            remaining_days = sorted(set(all_days_full) - set(all_done_dates))
            remaining_days_str = (
                f"{remaining_days[0]} → {remaining_days[-1]} ({len(remaining_days)} днів)"
                if remaining_days else "–"
            )
        else:
            skipped_days_str = "–"
            remaining_days_str = f"{start[:10]} → {end[:10]}"

        _p(f"▶ Починаємо: {total} тікерів (пропущено {len(done_tickers)} по старому маніфесту) | {start} → {end} | interval={interval} | next batch #{batch_idx:04d} | run={run_id}")
        _p(f"   workers: tickers={max_workers_tickers if parallel_tickers else 1}, chunks={max_workers_chunks if parallel_chunks else 0} | blue_ocean={include_blue_ocean} | day-manifest={use_day_manifest}")
        _p(f"   day-manifest: повністю готових тікерів пропущено = {len(fully_done)}")
        _p(f"   📉 пропущені дні (в межах window): {skipped_days_str}")
        _p(f"   📈 обробляються дні: {remaining_days_str}")
        _p(_mem_str(f"   {psutil_note} · старт · "))

    def _make_batch_path(idx: int) -> str:
        fname = f"{out_prefix}_{idx:04d}.csv" + (".gz" if gzip else "")
        return os.path.join(out_dir, fname)

    def _append_df_for_buffer(dfi: pd.DataFrame):
        nonlocal rows_in_buffer
        if dfi is None or dfi.empty:
            return
        buffer.append(dfi)
        rows_in_buffer += len(dfi)

    # NEW: миттєво оновлюємо in-memory кеш днів із нових рядків
    def _update_day_cache_from_df(dfi: pd.DataFrame):
        if not use_day_manifest or dfi is None or dfi.empty:
            return
        if "ticker" not in dfi.columns or "dt" not in dfi.columns:
            return
        tmp = dfi[["ticker", "dt"]].copy()
        tmp["date"] = tmp["dt"].astype(str).str.slice(0, 10)
        for tk, grp in tmp.groupby("ticker"):
            done_days_by_ticker.setdefault(str(tk).upper(), set()).update(grp["date"].astype(str).tolist())

    def _flush():
        nonlocal buffer, batch_idx, written_total, rows_in_buffer, last_flush_at
        if not buffer:
            return None
        path = _make_batch_path(batch_idx)
        while os.path.exists(path):
            batch_idx += 1
            path = _make_batch_path(batch_idx)

        if verbose:
            _p(_mem_str("   ⏳ перед flush · "))

        out = pd.concat(buffer, ignore_index=True)

        # Stable order & dedup
        if "dt" in out.columns:
            out = out.drop_duplicates(subset=["ticker","dt"], keep="last")
            out = out.sort_values(["ticker","dt"], kind="mergesort").reset_index(drop=True)

        tmp_path = path + ".tmp"
        if gzip:
            out.to_csv(tmp_path, index=False, compression={"method": "gzip", "compresslevel": 1})
        else:
            out.to_csv(tmp_path, index=False, compression=None)
        os.replace(tmp_path, path)

        written_total += len(out)
        if verbose:
            _p(f"💾 batch {batch_idx} → {len(out):,} рядків (всього {written_total:,} у цьому запуску) → {path}")

        # NEW: update day-manifest (та одразу оновити кеш у пам'яті)
        if use_day_manifest:
            if "dt" not in out.columns:
                for alt in ("datetime", "time", "t"):
                    if alt in out.columns:
                        out = out.rename(columns={alt: "dt"})
                        break
            _append_done_days_manifest(out_dir, out)
            _update_day_cache_from_df(out)
        else:
            if "ticker" in out.columns:
                _append_done_tickers_manifest(out_dir, out["ticker"].astype(str).unique().tolist())

        # (опційно) відмічати повністю закриті тікери у старому маніфесті, якщо він увімкнений
        if use_manifest and use_day_manifest:
            def _ticker_fully_done_local(tk: str) -> bool:
                s = done_days_by_ticker.get(str(tk).upper(), set())
                return bool(s) and all(dd in s for dd in all_days_str)
            fully_done_now = []
            for tk in out.get("ticker", pd.Series(dtype=str)).astype(str).str.upper().unique():
                if _ticker_fully_done_local(tk):
                    fully_done_now.append(tk)
            _append_done_tickers_manifest(out_dir, fully_done_now)

        batch_idx += 1
        buffer.clear()
        rows_in_buffer = 0
        del out
        gc.collect()
        last_flush_at = time.time()

        if verbose:
            _p(_mem_str("   ✅ після flush · "))
        return path

    def _progress_line(done: int) -> str:
        pct = (done / total * 100) if total else 100.0
        elapsed = time.time() - start_time
        per_item = elapsed / done if done > 0 else 0.0
        remaining = (total - done) * per_item if done > 0 else 0.0
        return f"{pct:6.2f}% | ETA {_fmt_eta(remaining)}"

    def _should_print(idx: int, is_error: bool, is_last: bool) -> bool:
        if is_error or is_last:
            return True
        if progress_every and progress_every > 0:
            return (idx % progress_every == 0)
        return True

    def _maybe_log_mem(idx: int):
        if not verbose:
            return
        if mem_log_every and mem_log_every > 0 and idx % mem_log_every == 0:
            _p(_mem_str("   • RAM · "))

    def _maybe_flush_by_thresholds(processed: int):
        if not buffer:
            return
        need_rows = (flush_when_rows is not None and rows_in_buffer >= flush_when_rows)
        need_time = (flush_every_seconds is not None and (time.time() - last_flush_at) >= flush_every_seconds)
        need_periodic = (save_every and processed % save_every == 0)
        if need_rows or need_time or need_periodic:
            _flush()

    # NEW: якщо за діапазон взагалі не прийшли дані — позначаємо дні як виконані
    def _mark_days_done(tk: str, days_list: List[pd.Timestamp]):
        if not use_day_manifest or not days_list:
            return
        df_mark = pd.DataFrame({
            "ticker": str(tk).upper(),
            "dt": [pd.Timestamp(d).strftime("%Y-%m-%d 00:00:00") for d in days_list],
        })
        _append_done_days_manifest(out_dir, df_mark)
        done_days_by_ticker.setdefault(str(tk).upper(), set()).update(
            [pd.Timestamp(d).strftime("%Y-%m-%d") for d in days_list]
        )

    # --- один діапазон: основний ендпоінт (+ Blue Ocean) ---
    def _fetch_range(tk: str, a: str, b: str):
        """ok=False, якщо будь-який виклик ВПАВ (429/5xx/мережа). Порожня відповідь 200 — не збій."""
        parts: List[pd.DataFrame] = []
        ok, why = True, None
        calls = [lambda: _one_call_v3_dt_fast(tk, a, b, interval, chart_type="ohlcv",
                                              include_boats=False, normalize_ohlcv=False)]
        if include_blue_ocean:
            calls.append(lambda: _one_call_blue_ocean_v3_dt_fast(tk, a, b, interval, chart_type="ohlcv",
                                                                 normalize_ohlcv=False))
        for call in calls:
            try:
                df = call()
            except FetchFailed as e:
                ok, why = False, str(e)
                continue
            if df is not None and not df.empty:
                parts.append(df)
        return a, b, parts, ok, why

    # --- per-ticker worker (respects day-manifest gaps) ---
    def _one(tk: str):
        try:
            # determine which days to fetch
            all_days = all_days_list
            if use_day_manifest:
                done_set = done_days_by_ticker.get(str(tk).upper(), set())
                todo_days = [d for d in all_days if d.strftime("%Y-%m-%d") not in done_set]
            else:
                todo_days = all_days

            if not todo_days:
                return tk, pd.DataFrame(), None

            # group missing days into ranges (capped by chunk_days)
            todo_ranges = _group_consecutive_dates(todo_days, max_span_days=max(1, int(chunk_days)))

            if parallel_chunks and len(todo_ranges) > 1:
                w = max(1, min(int(max_workers_chunks), len(todo_ranges)))
                with ThreadPoolExecutor(max_workers=w) as ex:
                    results = list(ex.map(lambda r: _fetch_range(tk, r[0], r[1]), todo_ranges))
            else:
                results = [_fetch_range(tk, a, b) for (a, b) in todo_ranges]

            parts: List[pd.DataFrame] = []
            ok_days, bad_days, failed = set(), set(), []
            for a, b, p, ok, why in results:
                days = {d.strftime("%Y-%m-%d") for d in _dates_between(a, b)}
                if ok:
                    parts.extend(p)
                    ok_days |= days
                else:
                    # діапазон впав: його дні НЕ можна вважати готовими, а частково прийшлі
                    # рядки цих днів відкидаємо — інакше _flush позначить день «готовим»
                    # і докачки (напр. Blue Ocean частини) вже ніколи не буде
                    parts.extend(p)
                    bad_days |= days
                    failed.append(f"{a[:10]}..{b[:10]}: {why}")
            err = ("; ".join(failed))[:600] if failed else None

            dfo = pd.DataFrame()
            if parts:
                dfo = pd.concat(parts, ignore_index=True)
                if "dt" in dfo.columns:
                    dfo = dfo.drop_duplicates(subset=["ticker","dt"], keep="last")
                    dfo = dfo.sort_values(["ticker","dt"], kind="mergesort").reset_index(drop=True)

                # завжди залишаємо 'ticker' та 'dt' (щоб маніфест працював)
                if keep_columns:
                    base = ["ticker", "dt"]
                    cols = base + [c for c in keep_columns if c in dfo.columns and c not in base]
                    cols = [c for c in cols if c in dfo.columns]
                    dfo = dfo[cols]

            present = set()
            if len(dfo) and "dt" in dfo.columns:
                day_s = dfo["dt"].astype(str).str.slice(0, 10)
                present = set(day_s)
                if bad_days:
                    dfo = dfo[~day_s.isin(bad_days)]

            # успішний діапазон без жодного бара за день = справді порожній день → готовий
            # (дні з барами позначить _flush після запису на диск)
            _mark_days_done(tk, [pd.Timestamp(d) for d in sorted(ok_days - present)])
            return tk, dfo, err
        except Exception as e:
            return tk, None, str(e)

    if parallel_tickers:
        with ThreadPoolExecutor(max_workers=max_workers_tickers) as ex:
            futs = {ex.submit(_one, tk): tk for tk in tickers}
            processed = 0
            for f in as_completed(futs):
                tk = futs[f]
                processed += 1
                is_last = (processed == total)
                try:
                    tk_, df, err = f.result()
                except Exception as e:
                    errors.append({"ticker": tk, "error": str(e)})
                    if _should_print(processed, True, is_last) and verbose:
                        _p(f"[{processed}/{total}] ✗ {tk}: {e} | {_progress_line(processed)}")
                    _maybe_log_mem(processed)
                    continue

                if err and df is not None and not df.empty:
                    # частковий збій: що прийшло — зберігаємо, збійні діапазони докачаються наступним запуском
                    errors.append({"ticker": tk_, "error": "PARTIAL: " + err})
                    if verbose:
                        _p(f"[{processed}/{total}] ⚠ {tk_}: частковий збій (буде докачано) | {err[:160]}")
                if err and (df is None or df.empty):
                    errors.append({"ticker": tk_, "error": err})
                    if _should_print(processed, True, is_last) and verbose:
                        _p(f"[{processed}/{total}] ✗ {tk_}: {err} | {_progress_line(processed)}")
                else:
                    if df is not None and not df.empty:
                        _append_df_for_buffer(df)
                        _update_day_cache_from_df(df)
                        if _should_print(processed, False, is_last) and verbose:
                            _p(f"[{processed}/{total}] ✓ {tk_}: {len(df):,} рядків | {_progress_line(processed)}")
                    else:
                        if _should_print(processed, False, is_last) and verbose:
                            _p(f"[{processed}/{total}] – {tk_}: empty | {_progress_line(processed)}")

                del df
                gc.collect()
                _maybe_log_mem(processed)
                _maybe_flush_by_thresholds(processed)
    else:
        for i, tk in enumerate(tickers, 1):
            is_last = (i == total)
            tk_, df, err = _one(tk)
            if err and df is not None and not df.empty:
                errors.append({"ticker": tk_, "error": "PARTIAL: " + err})
                if verbose:
                    _p(f"[{i}/{total}] ⚠ {tk_}: частковий збій (буде докачано) | {err[:160]}")
            if err and (df is None or df.empty):
                errors.append({"ticker": tk_, "error": err})
                if _should_print(i, True, is_last) and verbose:
                    _p(f"[{i}/{total}] ✗ {tk_}: {err} | {_progress_line(i)}")
            else:
                if df is not None and not df.empty:
                    _append_df_for_buffer(df)
                    _update_day_cache_from_df(df)
                    if _should_print(i, False, is_last) and verbose:
                        _p(f"[{i}/{total}] ✓ {tk_}: {len(df):,} рядків | {_progress_line(i)}")
                else:
                    if _should_print(i, False, is_last) and verbose:
                        _p(f"[{i}/{total}] – {tk_}: empty | {_progress_line(i)}")

            del df
            gc.collect()
            _maybe_log_mem(i)
            _maybe_flush_by_thresholds(i)

    _flush()
    total_elapsed = time.time() - start_time
    if verbose:
        _p(_mem_str("   🔚 фінал · "))
        _p(f"🏁 Готово. Нових рядків: {written_total:,} → {out_dir} | Час: {_fmt_eta(total_elapsed)}")
        if errors:
            _p(f"⚠️ Тікерів зі збоями API: {len(errors)}. Їхні дні НЕ позначені готовими — "
               f"перезапусти цю ж клітинку, вона докачає саме їх.")
    return pd.DataFrame(errors)


def rebuild_done_days_from_batches(out_dir: str, start_date_str: str, end_date_str: str,
                                   pattern: str = "intraday_*.csv.gz") -> dict:
    """Перебудовує done_days.csv ЛИШЕ з того, що реально лежить у батчах.

    Старий маніфест містив «готові» дні без жодного бара (збій API → порожньо → «готово»)
    і побиті рядки. Після перебудови день вважається готовим, лише якщо в батчах є бар
    цього (ticker, date). Решту fetch_intraday_v3_for_all_tickers докачає і сам позначить
    справді порожні дні (вихідні/свята) — але тільки після УСПІШНОЇ відповіді API.
    """
    import pyarrow.csv as pacsv
    import pyarrow.compute as pc
    import pyarrow as pa
    files = sorted(glob.glob(os.path.join(out_dir, pattern)))
    if not files:
        raise FileNotFoundError(f"Немає {pattern} у {out_dir}")
    d0, d1 = str(start_date_str)[:10], str(end_date_str)[:10]
    pairs = set()
    for k, fp in enumerate(files, 1):
        rdr = pacsv.open_csv(fp, convert_options=pacsv.ConvertOptions(
            include_columns=["ticker", "dt"],
            column_types={"ticker": pa.string(), "dt": pa.string()}))
        for rb in rdr:
            t = pa.Table.from_batches([rb])
            g = pa.table({"ticker": pc.utf8_upper(pc.utf8_trim_whitespace(t["ticker"])),
                          "date": pc.utf8_slice_codeunits(t["dt"], 0, 10)}) \
                  .group_by(["ticker", "date"]).aggregate([])
            pairs.update(zip(g["ticker"].to_pylist(), g["date"].to_pylist()))
        if k % 100 == 0:
            print(f"  [{k}/{len(files)}] пар (ticker,date): {len(pairs):,}", flush=True)
    pairs = {(tk, dd) for tk, dd in pairs
             if tk and dd and re.fullmatch(r"\d{4}-\d\d-\d\d", dd) and d0 <= dd <= d1}
    path = _manifest_days_path(out_dir)
    old_n = 0
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            old_n = max(0, sum(1 for _ in f) - 1)
        os.replace(path, path + ".bak")
    with _MANIFEST_LOCK, open(path, "w", encoding="utf-8", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(["ticker", "date"])
        for tk, dd in sorted(pairs):
            wr.writerow([tk, dd])
    print(f"✅ done_days.csv: було {old_n:,} → стало {len(pairs):,} (старий збережено як .bak)")
    done = {}
    for tk, dd in pairs:
        done.setdefault(tk, set()).add(dd)
    return done


In [15]:
TICKERS

,ticker,value,lvl3,market_cap
0,A,68669.72,Medical Equipment & Devices,49300.0
1,AA,935733.26,Metals & Mining,10342.0
2,AAAU,459786.34,NaN,NaN
3,AACG,26.70,Software,27.0
4,AADX,57173.51,Aerospace & Defense,1922.0
...,...,...,...,...
7093,ZVOL,6670.50,NaN,NaN
7094,ZVRA,4056.64,Biotech & Pharma,610.0
7095,ZWS,999.25,Machinery,7396.0
7096,ZYBT,3836.18,Biotech & Pharma,38.0


In [16]:
import pandas as pd

# список тикерів
tickers = [
    'ARKK', 'COPX', 'DRAM', 'FCX', 'FXI', 'GDX', 'IBIT', 'IGV', 'ITA', 'IWM',
    'KRE', 'KWEB', 'NASA', 'QQQ', 'SLV', 'SOXX', 'SPY', 'UNG', 'URA', 'XBI',
    'XLB', 'XLE', 'XLF', 'XLP', 'XLU', 'XLV', 'XOP', 'XRT',
]


# створення DataFrame
df = pd.DataFrame(tickers, columns=['ticker'])

print(df)


   ticker
0    ARKK
1    COPX
2    DRAM
3     FCX
4     FXI
5     GDX
6    IBIT
7     IGV
8     ITA
9     IWM
10    KRE
11   KWEB
12   NASA
13    QQQ
14    SLV
15   SOXX
16    SPY
17    UNG
18    URA
19    XBI
20    XLB
21    XLE
22    XLF
23    XLP
24    XLU
25    XLV
26    XOP
27    XRT


In [17]:
# Одноразово ПЕРЕД докачкою (клітинки з fetch_intraday_v3_for_all_tickers): перебудувати
# done_days.csv з реальних батчів. Старі маніфести містили «готові» дні без жодного бара
# (збій API трактувався як «порожньо»), через що ці дні не докачувались ніколи.
# Постав True, запусти, і знову постав False.
REBUILD_DONE_DAYS = False
if REBUILD_DONE_DAYS:
    rebuild_done_days_from_batches(str(BATCH_DIR), start_date_str, end_date_str)
    rebuild_done_days_from_batches(str(ETF_DIR), start_date_str, end_date_str)


In [18]:
keep = ["ticker","dt","o","h","l","c","v"]

errs = fetch_intraday_v3_for_all_tickers(
    tickers_df=TICKERS,
    start= start,
    end= end,
    interval=1,
    chunk_days=30,
    parallel_chunks=True,
    max_workers_chunks=16,
    parallel_tickers=True,
    max_workers_tickers=16,
    save_every=50,
    out_dir=str(BATCH_DIR),
    out_prefix="intraday",
    gzip=True,
    keep_columns=keep,
    verbose=True,
    include_blue_ocean=True,
)

# Збої API більше не маскуються під «порожні дні»: усе, що тут перелічено, НЕ позначено
# готовим. Запускай клітинку повторно, доки errs не стане порожнім.
if errs is not None and len(errs):
    print(f"⚠️ {len(errs)} тікерів зі збоями — перезапусти клітинку:")
    print(errs.head(20).to_string())


▶ Починаємо: 7098 тікерів (пропущено 0 по старому маніфесту) | 2026-08-01 00:00:00 → 2026-11-01 23:59:59 | interval=1 | next batch #21940 | run=d2165d2c


   workers: tickers=16, chunks=16 | blue_ocean=True | day-manifest=True


   day-manifest: повністю готових тікерів пропущено = 0


   📉 пропущені дні (в межах window): 2026-08-01 → 2026-10-31 (92 днів)


   📈 обробляються дні: 2026-11-01 → 2026-11-01 (1 днів)


   psutil · старт · 🧠 RAM: 340.4 MB


... (1,275 outputs left out)

💾 batch 22081 → 275 рядків (всього 112,861 у цьому запуску) → C:\datum-api-examples-main\OriON\CRACEN\work\CRACEN_batch\intraday_22081.csv.gz


   ✅ після flush · 🧠 RAM: 342.8 MB


   🔚 фінал · 🧠 RAM: 342.8 MB


🏁 Готово. Нових рядків: 112,861 → C:\datum-api-examples-main\OriON\CRACEN\work\CRACEN_batch | Час: 21:32


In [19]:
# ================== FAST LOCAL POST-PROCESSING (no API calls) ==================
# Читання батчів робить pyarrow (нативний багатопотоковий CSV-парсер) замість
# pandas.read_csv, а premarket-фільтр і денні open/close рахуються за ОДИН прохід
# по gz-файлах замість двох.
import os, gc, glob, time, threading
from typing import Optional, List, Tuple

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.csv as pacsv
import pyarrow.compute as pc
import pyarrow.dataset as pads
from concurrent.futures import ThreadPoolExecutor

_INTRADAY_COLS = ["ticker", "dt", "o", "h", "l", "c", "v"]

# Парсимо у float64, щоб денні open/close збігалися з попередньою (pandas) реалізацією
# біт у біт; premarket-рядки перед записом кастимо назад у float32 — стара схема на диску.
_INTRADAY_TYPES = {
    "ticker": pa.string(), "dt": pa.string(),
    "o": pa.float64(), "h": pa.float64(), "l": pa.float64(),
    "c": pa.float64(), "v": pa.float64(),
}
_PREMARKET_SCHEMA = pa.schema([
    ("ticker", pa.string()), ("dt", pa.string()),
    ("o", pa.float32()), ("h", pa.float32()), ("l", pa.float32()),
    ("c", pa.float32()), ("v", pa.float32()),
])


# ---------- спільні хелпери для мерджів по тікеру (використовують клітинки 28/40/44) ----------
def _ticker_codes(s: pd.Series):
    """Факторизує колонку тікерів і нормалізує (strip+upper) ЛИШЕ унікальні значення.

    Повертає (codes, categories); codes[i] == -1 означає null. Сенс у тому, щоб
    strip/upper і словниковий пошук виконувались над ~10^3 унікальними тікерами,
    а не над ~10^6 рядками батча.
    """
    cat = s.astype("category")
    cats = pd.Index(cat.cat.categories.astype(str)).str.strip().str.upper()
    return cat.cat.codes.to_numpy(), cats


def _expand(values: np.ndarray, codes: np.ndarray, fill):
    """Розгортає масив «на категорію» назад «на рядок»; де codes == -1 ставить fill."""
    out = np.empty(codes.shape, dtype=values.dtype if values.dtype != object else object)
    valid = codes >= 0
    out[valid] = values[codes[valid]]
    if (~valid).any():
        out[~valid] = fill
    return out


def _map_by_ticker(codes: np.ndarray, cats: pd.Index, mapping: dict, numeric: bool):
    vals = cats.map(mapping)
    if numeric:
        vals = pd.to_numeric(pd.Series(vals), errors="coerce").to_numpy(dtype="float64")
        return _expand(vals, codes, np.nan)
    vals = np.asarray(pd.Series(vals).astype(object).to_numpy(), dtype=object)
    return _expand(vals, codes, None)


# ---------- читання intraday-батчів ----------
def _open_intraday_csv(path: str, block_size: int):
    return pacsv.open_csv(
        path,
        read_options=pacsv.ReadOptions(use_threads=True, block_size=block_size),
        convert_options=pacsv.ConvertOptions(
            include_columns=_INTRADAY_COLS, column_types=_INTRADAY_TYPES
        ),
    )


def _time_date_mask(dt_arr, start_time, end_time, date_min, date_max):
    t = pc.utf8_slice_codeunits(dt_arr, 11, 19)
    mask = pc.and_(pc.greater_equal(t, start_time), pc.less_equal(t, end_time))
    if date_min is not None or date_max is not None:
        d = pc.utf8_slice_codeunits(dt_arr, 0, 10)
        if date_min is not None:
            mask = pc.and_(mask, pc.greater_equal(d, date_min))
        if date_max is not None:
            mask = pc.and_(mask, pc.less_equal(d, date_max))
    return mask


def _oc_from_table(tbl: pa.Table) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Перший open у вікні 09:30–09:35 і останній close у 15:55–15:59 на (ticker, date)."""
    OPEN_FROM, OPEN_TO = "09:30", "09:35"
    CLOSE_FROM, CLOSE_TO = "15:55", "15:59"

    hhmm = pc.utf8_slice_codeunits(tbl.column("dt"), 11, 16)
    m_o = pc.and_(pc.and_(pc.greater_equal(hhmm, OPEN_FROM), pc.less_equal(hhmm, OPEN_TO)),
                  pc.is_valid(tbl.column("o")))
    m_c = pc.and_(pc.and_(pc.greater_equal(hhmm, CLOSE_FROM), pc.less_equal(hhmm, CLOSE_TO)),
                  pc.is_valid(tbl.column("c")))

    def _reduce(mask, val_col, keep):
        sub = tbl.select(["ticker", "dt", val_col]).filter(mask)
        if sub.num_rows == 0:
            return pd.DataFrame(columns=["ticker", "date", "dt", val_col])
        d = sub.to_pandas()
        d["date"] = d["dt"].str.slice(0, 10)
        d = d.sort_values("dt", kind="stable").drop_duplicates(["ticker", "date"], keep=keep)
        return d[["ticker", "date", "dt", val_col]]

    return _reduce(m_o, "o", "first"), _reduce(m_c, "c", "last")


def _finalize_oc(open_parts: List[pd.DataFrame], close_parts: List[pd.DataFrame]) -> pd.DataFrame:
    opens = (pd.concat(open_parts, ignore_index=True) if open_parts
             else pd.DataFrame(columns=["ticker", "date", "dt", "o"]))
    closes = (pd.concat(close_parts, ignore_index=True) if close_parts
              else pd.DataFrame(columns=["ticker", "date", "dt", "c"]))
    if len(opens):
        opens = opens.sort_values("dt", kind="stable").drop_duplicates(["ticker", "date"], keep="first")
    if len(closes):
        closes = closes.sort_values("dt", kind="stable").drop_duplicates(["ticker", "date"], keep="last")
    opens = opens[["ticker", "date", "o"]].rename(columns={"o": "open"})
    closes = closes[["ticker", "date", "c"]].rename(columns={"c": "close"})

    out = (opens.merge(closes, on=["ticker", "date"], how="outer")
                .sort_values(["ticker", "date"], kind="stable").reset_index(drop=True))
    out["open"] = pd.to_numeric(out["open"], errors="coerce").round(2)
    out["close"] = pd.to_numeric(out["close"], errors="coerce").round(2)
    return out


def collect_premarket_and_oc(
    folder: str,
    out_path: str,
    oc_out_path: Optional[str] = None,
    pattern: str = "intraday_*.csv.gz",
    start_time: str = "00:00:00",
    end_time: str = "23:59:00",
    *,
    date_min: Optional[str] = None,     # 'YYYY-MM-DD' inclusive
    date_max: Optional[str] = None,     # 'YYYY-MM-DD' inclusive
    build_oc: bool = True,
    max_workers: int = 4,
    block_size: int = 64 << 20,
    compression: str = "snappy",
    sort_chunk: bool = True,
    verbose: bool = True,
):
    """ОДИН прохід по {folder}/{pattern}: пише відфільтровані рядки у out_path і
    (опційно) денні open/close у oc_out_path.

    Замінює пару collect_premarket_by_day_stream + build_oc_from_intraday_folder,
    які раніше читали й розтискали ті самі gz-файли двічі.

    Порядок рядків МІЖ файлами недетермінований (файли читаються паралельно).
    Це безпечно: усі наступні кроки — мерджі по ключах, а фінальний файл
    фізично сортується у клітинці 47. Для точного старого порядку — max_workers=1.
    """
    files = sorted(glob.glob(os.path.join(folder, pattern)))
    if not files:
        raise FileNotFoundError(f"Не знайдено файлів {pattern} у {folder}")

    if date_min is not None:
        date_min = str(date_min)[:10]
    if date_max is not None:
        date_max = str(date_max)[:10]
    if date_min is not None and date_max is not None and date_min > date_max:
        date_min, date_max = date_max, date_min

    if os.path.exists(out_path):
        os.remove(out_path)

    t0 = time.time()
    lock = threading.Lock()
    state = {"writer": None, "in": 0, "out": 0, "files": 0}
    open_parts: List[pd.DataFrame] = []
    close_parts: List[pd.DataFrame] = []

    def _handle(path: str):
        reader = _open_intraday_csv(path, block_size)
        loc_open, loc_close = [], []
        n_in = n_out = 0
        while True:
            try:
                rb = reader.read_next_batch()
            except StopIteration:
                break
            tbl = pa.Table.from_batches([rb])
            n_in += tbl.num_rows

            if build_oc:
                o_df, c_df = _oc_from_table(tbl)
                if len(o_df):
                    loc_open.append(o_df)
                if len(c_df):
                    loc_close.append(c_df)

            cut = tbl.filter(_time_date_mask(tbl.column("dt"), start_time, end_time,
                                             date_min, date_max))
            if cut.num_rows:
                cut = cut.cast(_PREMARKET_SCHEMA)
                if sort_chunk:
                    cut = cut.sort_by([("ticker", "ascending"), ("dt", "ascending")])
                n_out += cut.num_rows
                with lock:
                    if state["writer"] is None:
                        state["writer"] = pq.ParquetWriter(out_path, cut.schema,
                                                           compression=compression)
                    state["writer"].write_table(cut)

        with lock:
            state["in"] += n_in
            state["out"] += n_out
            state["files"] += 1
            open_parts.extend(loc_open)
            close_parts.extend(loc_close)
            if verbose:
                print(f"  [{state['files']}/{len(files)}] {os.path.basename(path)}  "
                      f"in={state['in']:,} out={state['out']:,} "
                      f"elapsed={time.time()-t0:,.1f}s", flush=True)

    rng_str = f"  date=[{date_min or '…'}..{date_max or '…'}]" if (date_min or date_max) else ""
    if verbose:
        print(f"▶️ START: {folder}/{pattern} → {out_path}  files={len(files)} "
              f"workers={max_workers}{rng_str}", flush=True)
    try:
        if max_workers and max_workers > 1:
            with ThreadPoolExecutor(max_workers=max_workers) as ex:
                list(ex.map(_handle, files))
        else:
            for p in files:
                _handle(p)
    finally:
        if state["writer"] is not None:
            state["writer"].close()

    df_oc = None
    if build_oc:
        df_oc = _finalize_oc(open_parts, close_parts)
        if oc_out_path:
            os.makedirs(os.path.dirname(oc_out_path) or ".", exist_ok=True)
            df_oc.to_parquet(oc_out_path, index=False)

    if verbose:
        print(f"✅ DONE  in={state['in']:,} → out={state['out']:,}  to {out_path}  "
              f"in {time.time()-t0:,.1f}s", flush=True)
    return out_path, df_oc


def collect_premarket_by_day_stream(
    folder: str,
    out_path: str = "CRACEN/work/premarket_rows.parquet",
    pattern: str = "intraday_*.csv.gz",
    start_time: str = "00:00:00",
    end_time: str = "23:59:00",
    *,
    date_min: Optional[str] = None,
    date_max: Optional[str] = None,
    aggregate: bool = False,
    chunksize: int = 200_000,          # лишено для сумісності виклику; pyarrow керує блоками сам
    compression: str = "snappy",
    log_every: int = 5,                # лишено для сумісності виклику
    sort_chunk: bool = True,
    max_workers: int = 4,
    verbose: bool = True,
):
    """Тонка обгортка над collect_premarket_and_oc (build_oc=False) — щоб старі
    виклики працювали без змін."""
    if aggregate:
        raise NotImplementedError("aggregate=True у цьому пайплайні не використовується")
    path, _ = collect_premarket_and_oc(
        folder, out_path, oc_out_path=None, pattern=pattern,
        start_time=start_time, end_time=end_time,
        date_min=date_min, date_max=date_max, build_oc=False,
        max_workers=max_workers, compression=compression,
        sort_chunk=sort_chunk, verbose=verbose,
    )
    return path


In [20]:
# Один прохід по gz-батчах: одразу і premarket-рядки, і денні open/close.
_, df_oc = collect_premarket_and_oc(
    folder=str(BATCH_DIR),
    out_path=str(WORK_DIR / "premarket_rows.parquet"),
    oc_out_path=str(WORK_DIR / "oc.parquet"),
    start_time="00:00:00",
    end_time="23:59:00",
    date_min=start_date_str,
    date_max=end_date_str,
    build_oc=True,
    max_workers=4,
)
print(df_oc.head())


▶️ START: C:\datum-api-examples-main\OriON\CRACEN\work\CRACEN_batch/intraday_*.csv.gz → C:\datum-api-examples-main\OriON\CRACEN\work\premarket_rows.parquet  files=22081 workers=4  date=[2026-08-01..2026-11-01]


  [1/22081] intraday_0003.csv.gz  in=727,885 out=0 elapsed=0.7s


  [2/22081] intraday_0004.csv.gz  in=1,470,913 out=0 elapsed=0.7s


  [3/22081] intraday_0002.csv.gz  in=2,357,625 out=0 elapsed=0.8s


  [4/22081] intraday_0001.csv.gz  in=3,333,452 out=0 elapsed=0.8s


  [5/22081] intraday_0005.csv.gz  in=4,207,443 out=0 elapsed=1.5s


... (22,074 outputs left out)

  [22080/22081] intraday_9999.csv.gz  in=492,996,177 out=111,611,206 elapsed=288.3s


  [22081/22081] intraday_9998.csv.gz  in=493,009,508 out=111,611,206 elapsed=288.3s


✅ DONE  in=493,009,508 → out=111,611,206  to C:\datum-api-examples-main\OriON\CRACEN\work\premarket_rows.parquet  in 301.8s


  ticker        date    open   close
0      A  2025-10-28  145.74  145.85
1      A  2025-10-29  145.52  143.23
2      A  2025-10-30  142.93  143.39
3      A  2025-10-31  143.10  146.36
4      A  2025-11-03  145.07  144.72


In [21]:
def build_oc_from_intraday_folder(
    folder: str,
    out_path: Optional[str] = None,
    chunksize: int = 250_000,      # лишено для сумісності виклику
    *,
    max_workers: int = 4,
) -> pd.DataFrame:
    """Денні open (перший бар 09:30–09:35) і close (останній бар 15:55–15:59).

    Раніше кожен відібраний рядок проходив через itertuples + оновлення python-словника;
    тепер редукція повністю векторна (sort + drop_duplicates у pyarrow/pandas).
    Клітинка 19 уже рахує це за один прохід — ця функція лишається як самостійний шлях.
    """
    files = sorted(
        glob.glob(os.path.join(folder, "intraday_*.csv")) +
        glob.glob(os.path.join(folder, "intraday_*.csv.gz"))
    )
    if not files:
        raise FileNotFoundError(f"Не знайдено intraday_*.csv[.gz] у: {folder}")

    open_parts, close_parts = [], []
    lock = threading.Lock()

    def _handle(path):
        reader = _open_intraday_csv(path, 64 << 20)
        lo, lc = [], []
        while True:
            try:
                rb = reader.read_next_batch()
            except StopIteration:
                break
            o_df, c_df = _oc_from_table(pa.Table.from_batches([rb]))
            if len(o_df):
                lo.append(o_df)
            if len(c_df):
                lc.append(c_df)
        with lock:
            open_parts.extend(lo)
            close_parts.extend(lc)

    if max_workers and max_workers > 1:
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            list(ex.map(_handle, files))
    else:
        for p in files:
            _handle(p)

    out = _finalize_oc(open_parts, close_parts)

    if out_path is None:
        out_path = os.path.join(folder, "oc.parquet")
    if not out_path.endswith(".parquet"):
        out_path += ".parquet"
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    out.to_parquet(out_path, index=False)
    return out


In [22]:
# oc.parquet уже побудовано в клітинці 19 (той самий прохід по файлах).
# Перебудовуємо лише якщо його немає.
_oc_path = str(WORK_DIR / "oc.parquet")
if os.path.exists(_oc_path):
    df_oc = pd.read_parquet(_oc_path)
else:
    df_oc = build_oc_from_intraday_folder(folder=str(BATCH_DIR), out_path=_oc_path)
print(df_oc.head())


  ticker        date    open   close
0      A  2025-10-28  145.74  145.85
1      A  2025-10-29  145.52  143.23
2      A  2025-10-30  142.93  143.39
3      A  2025-10-31  143.10  146.36
4      A  2025-11-03  145.07  144.72


In [23]:
import os
import gc
import time
import pandas as pd
import numpy as np

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception:
    pq = None


def _prep_oc_prev_close(df_oc: pd.DataFrame) -> pd.DataFrame:
    """['ticker','date','open','prev_close','same_day_close'], date — звичайний рядок 'YYYY-MM-DD'.

    'prev_close' — close ПОПЕРЕДНЬОГО торгового дня (до 16:01).
    'same_day_close' — close ЦЬОГО Ж дня (від 16:01, коли сесія вже
    закрилась о 16:00 і референс-клоуз має перерахуватись на
    сьогоднішній)."""
    oc = df_oc[["ticker", "date", "open", "close"]].copy()
    oc["ticker"] = oc["ticker"].astype(str)
    oc["date"] = oc["date"].astype(str).str.slice(0, 10)
    oc = oc.sort_values(["ticker", "date"], kind="stable")
    oc["prev_close"] = oc.groupby("ticker", sort=False)["close"].shift(1)
    oc = oc.rename(columns={"close": "same_day_close"})
    return oc[["ticker", "date", "open", "prev_close", "same_day_close"]].reset_index(drop=True)


def _rebase_prev_close_after_session_close(df: pd.DataFrame,
                                           dt_col: str = "dt",
                                           prev_close_col: str = "prev_close",
                                           same_day_close_col: str = "same_day_close",
                                           close_boundary: str = "16:01") -> "pd.Series":
    """Від close_boundary (16:01, бо сесія закривається о 16:00) референс-клоуз
    кожного стака перераховується на клоуз ЦЬОГО Ж дня замість
    вчорашнього (інакше з 16:01 до півночі все й далі рахувалось би
    відносно вчорашнього клоуза). Час береться зрізом рядка 'dt'
    (без парсингу дат, як і решта пайплайну)."""
    time_str = df[dt_col].astype(str).str.slice(11, 16)
    after_close = time_str >= close_boundary
    return np.where(after_close, df[same_day_close_col], df[prev_close_col])


def _date_from_dt(s: pd.Series) -> pd.Series:
    """'YYYY-MM-DD' з 'dt'. Для рядкового dt це зріз (C-рівень), без парсингу дат."""
    if pd.api.types.is_string_dtype(s) or s.dtype == object:
        return s.astype(str).str.slice(0, 10)
    return s.dt.strftime("%Y-%m-%d")


def enrich_with_day_open_and_prev_close(df_intraday: pd.DataFrame,
                                        df_oc: pd.DataFrame,
                                        _oc_prepared: Optional[pd.DataFrame] = None,
                                        sort_out: bool = False) -> pd.DataFrame:
    """
    Додає до кожного рядка інтрадею 'open' дня та 'prev_close' — референс-клоуз,
    відносно якого рахується Stack%.
    Очікує в df_intraday: ['ticker','dt',...]
              в df_oc:       ['ticker','date','open','close']

    Референс-клоуз перераховується о 16:01: до цього часу — клоуз попереднього
    дня, від 16:01 (сесія вже закрилась о 16:00) — клоуз ЦЬОГО Ж дня. Інакше з
    16:01 до півночі рядки й далі рахувались б відносно вчорашнього клоуза.

    Прискорення проти попередньої версії:
      - 'dt' лишається рядком: ні pd.to_datetime, ні .dt.date (це були python-об'єкти
        на кожен рядок — найдорожча операція в усьому пайплайні);
      - prev_close рахується ОДИН раз (передається через _oc_prepared), а не на кожен батч;
      - без per-batch sort_values (фінальне сортування робить клітинка 47).
    """
    right = _oc_prepared if _oc_prepared is not None else _prep_oc_prev_close(df_oc)
    out = df_intraday
    out = out[out["dt"].notna()]
    out = out.assign(date=_date_from_dt(out["dt"]))
    out = out.merge(right, on=["ticker", "date"], how="left")
    out["prev_close"] = _rebase_prev_close_after_session_close(out)
    out = out.drop(columns=["same_day_close"])
    if sort_out:
        out = out.sort_values(["ticker", "dt"], kind="stable").reset_index(drop=True)
    return out


def merge_intraday_and_oc_to_file(
    intraday_parquet_path: str,
    oc_parquet_path: str,
    out_path: str,
    batch_rows: int = 2_000_000,
    log_every_batches: int = 5,
    sort_out: bool = False,
):
    """
    Читає великий Parquet батчами і мерджить денні open/prev_close з oc.parquet.
    """
    if pq is None:
        raise RuntimeError("Потрібен pyarrow (pyarrow.parquet).")

    t0 = time.time()

    df_oc = pd.read_parquet(oc_parquet_path)
    missing = {"ticker", "date", "open", "close"} - set(df_oc.columns)
    if missing:
        raise KeyError(f"OC-файл не містить колонок: {sorted(missing)}")
    oc_prepared = _prep_oc_prev_close(df_oc)      # рахуємо ОДИН раз, не на кожен батч

    if os.path.exists(out_path):
        os.remove(out_path)
    if not out_path.endswith((".csv", ".gz", ".parquet")):
        raise ValueError("Підтримані формати: .csv, .csv.gz, .parquet")

    parquet_writer = None
    total_rows = 0

    pf = pq.ParquetFile(intraday_parquet_path)
    cols = ["ticker", "dt", "o", "h", "l", "c", "v"]

    print(f"▶️ START: merge → {out_path} | batch_rows≈{batch_rows:,}", flush=True)

    for batch_idx, batch in enumerate(pf.iter_batches(batch_size=batch_rows, columns=cols), 1):
        df_batch = batch.to_pandas()
        df_enriched = enrich_with_day_open_and_prev_close(
            df_batch, None, _oc_prepared=oc_prepared, sort_out=sort_out)

        if out_path.endswith(".parquet"):
            table = pa.Table.from_pandas(df_enriched, preserve_index=False)
            if parquet_writer is None:
                parquet_writer = pq.ParquetWriter(out_path, table.schema, compression="snappy")
            parquet_writer.write_table(table)
        else:
            df_enriched.to_csv(
                out_path, index=False, mode="a", header=(total_rows == 0),
                compression="infer" if out_path.endswith(".gz") else None,
            )

        total_rows += len(df_enriched)
        if batch_idx % log_every_batches == 0:
            print(f"[batch {batch_idx:>4}] rows={total_rows:,}  "
                  f"elapsed={time.time()-t0:,.1f}s", flush=True)

    if parquet_writer is not None:
        parquet_writer.close()

    print(f"✅ DONE  rows={total_rows:,}  -> {out_path}  in {time.time()-t0:,.1f}s", flush=True)


def merge_intraday_and_oc_in_folder(
    folder: str = "CRACEN",
    intraday_filename: str = "premarket_rows.parquet",
    oc_filename: str = "oc.parquet",
    out_filename: str = "premarket_rows_enriched.parquet",
    batch_rows: int = 2_000_000,
    log_every_batches: int = 5,
) -> str:
    intraday_path = os.path.join(folder, intraday_filename)
    oc_path = os.path.join(folder, oc_filename)
    out_path = os.path.join(folder, out_filename)

    if not os.path.exists(intraday_path):
        raise FileNotFoundError(f"Не знайдено інтрадей: {intraday_path}")
    if not os.path.exists(oc_path):
        raise FileNotFoundError(f"Не знайдено OC-файл: {oc_path}")
    os.makedirs(folder, exist_ok=True)

    merge_intraday_and_oc_to_file(
        intraday_parquet_path=intraday_path,
        oc_parquet_path=oc_path,
        out_path=out_path,
        batch_rows=batch_rows,
        log_every_batches=log_every_batches,
    )
    return out_path


In [24]:
# ── ЗЛИТО В ОДИН ПРОХІД ─────────────────────────────────────────────
# Цей крок тепер робить build_final_in_one_pass у комірці 46: весь файл
# читається й пишеться ОДИН раз замість п'яти. Старий виклик лишено
# закоментованим — щоб відкотити, розкоментуй його і прибери виклик у 46.
# out_path = merge_intraday_and_oc_in_folder(folder=str(WORK_DIR))
# print("Результат у файлі:", out_path)


In [25]:
import os, time, gc
import pandas as pd
import numpy as np
from typing import Optional

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception:
    pq = None


def add_pct_vs_prev_close_parquet(
    input_path: str,
    output_path: str,
    batch_rows: int = 2_000_000,
    round_to: Optional[int] = None,
    log_every: int = 5,
    value_col: str = "o",        # ← що порівнюємо з prev_close (дефолт: open)
    out_col: str = "Stack%",     # назва колонки з відсотком
) -> str:
    """
    Потоково читає великий Parquet, додає колонку:
        out_col = (value_col / prev_close - 1) * 100
    Обчислення в numpy (без проміжних pandas-Series і .replace), без gc на кожен батч.
    """
    if pq is None:
        raise RuntimeError("Потрібен pyarrow (pyarrow.parquet).")

    t0 = time.time()
    if os.path.exists(output_path):
        os.remove(output_path)

    pf = pq.ParquetFile(input_path)
    parquet_writer = None
    total = 0

    print(f"▶️ START  file={input_path}  →  {output_path}  "
          f"batch_rows≈{batch_rows:,}  using={value_col}", flush=True)

    for batches, batch in enumerate(pf.iter_batches(batch_size=batch_rows), 1):
        df = batch.to_pandas()

        for c in ("prev_close", value_col):
            if c not in df.columns:
                raise KeyError(f"У вхідному файлі немає колонки '{c}'.")

        num = pd.to_numeric(df[value_col], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
        den = pd.to_numeric(df["prev_close"], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
        with np.errstate(divide="ignore", invalid="ignore"):
            pct = np.where(den == 0, np.nan, num / den - 1.0) * 100.0   # 0 → NaN, без ділення на 0
        if round_to is not None:
            pct = np.round(pct, int(round_to))

        df[out_col] = pct

        table = pa.Table.from_pandas(df, preserve_index=False)
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(output_path, table.schema, compression="snappy")
        parquet_writer.write_table(table)

        total += len(df)
        if batches % log_every == 0:
            print(f"[batch {batches:>4}] rows={total:,}  elapsed={time.time()-t0:,.1f}s", flush=True)

    if parquet_writer is not None:
        parquet_writer.close()

    print(f"✅ DONE  rows={total:,}  →  {output_path}  in {time.time()-t0:,.1f}s", flush=True)
    return output_path


def add_pct_vs_prev_close_in_folder(
    folder: str = "CRACEN",
    input_filename: str = "premarket_rows_enriched.parquet",
    output_filename: str = "premarket_rows_enriched_with_pct.parquet",
    batch_rows: int = 2_000_000,
    round_to: Optional[int] = 2,
    log_every: int = 5,
    value_col: str = "o",          # ← open vs prev_close
    out_col: str = "Stack%",       # назва нової колонки
) -> str:
    """Обгортка під твою папку CRACEN."""
    os.makedirs(folder, exist_ok=True)
    src = os.path.join(folder, input_filename)
    dst = os.path.join(folder, output_filename)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Не знайдено вхідний файл: {src}")
    return add_pct_vs_prev_close_parquet(
        input_path=src,
        output_path=dst,
        batch_rows=batch_rows,
        round_to=round_to,
        log_every=log_every,
        value_col=value_col,
        out_col=out_col,
    )


In [26]:
# ── ЗЛИТО В ОДИН ПРОХІД ─────────────────────────────────────────────
# Цей крок тепер робить build_final_in_one_pass у комірці 46: весь файл
# читається й пишеться ОДИН раз замість п'яти. Старий виклик лишено
# закоментованим — щоб відкотити, розкоментуй його і прибери виклик у 46.
# out_path = add_pct_vs_prev_close_in_folder(folder=str(WORK_DIR))
# print("Готово:", out_path)


In [27]:
import os
import pandas as pd
from typing import Optional

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception:
    pa = pq = None


def _read_mapping(mapping_path: str) -> pd.DataFrame:
    """
    Read ticker->bench mapping from csv/xlsx and return ['ticker','bench'].
    """
    ext = os.path.splitext(mapping_path)[1].lower()
    if ext in (".xlsx", ".xls"):
        df = pd.read_excel(mapping_path)
    else:
        try:
            df = pd.read_csv(mapping_path, sep=None, engine="python")
        except Exception:
            df = pd.read_csv(mapping_path)

    cols = {c.lower().strip(): c for c in df.columns}
    tcol = cols.get("ticker") or cols.get("symbol") or list(df.columns)[0]
    bcol = cols.get("bench") or cols.get("benchmark") or cols.get("etf") or list(df.columns)[1]

    out = df[[tcol, bcol]].rename(columns={tcol: "ticker", bcol: "bench"}).dropna()
    out["ticker"] = out["ticker"].astype(str).str.upper().str.strip()
    out["bench"] = out["bench"].astype(str).str.upper().str.strip()
    out = out[(out["ticker"] != "") & (out["bench"] != "")]
    return out.drop_duplicates(ignore_index=True)


def corr_beta_from_mapping_to_parquet(*args, **kwargs):
    """
    Deprecated API path removed.
    corr/beta is now computed locally in cell 12 and written in cell 13/28.
    """
    raise RuntimeError(
        "corr_beta_from_mapping_to_parquet is deprecated. "
        "Use local calculation in cell 12 and parquet output in cell 13/28."
    )



In [28]:
import os
import pandas as pd

out_path = str(WORK_DIR / "corr_beta_pairs.parquet")

# If parquet already exists from cell 13, keep it.
# Otherwise rebuild it from x_to_best_etf_by_lvl3.csv (no API calls).
if not os.path.exists(out_path):
    src_csv = str(WORK_DIR / "x_to_best_etf_by_lvl3.csv")
    if not os.path.exists(src_csv):
        raise FileNotFoundError(f"Neither {out_path} nor {src_csv} exists")

    d = pd.read_csv(src_csv)
    need = ["x_ticker", "best_y_ticker", "best_corr", "beta_with_best"]
    miss = [c for c in need if c not in d.columns]
    if miss:
        raise KeyError(f"Missing columns in {src_csv}: {miss}")

    d = d.rename(columns={
        "best_y_ticker": "y_ticker",
        "best_corr": "corr",
        "beta_with_best": "beta",
    })[["x_ticker", "y_ticker", "corr", "beta"]]
    d.to_parquet(out_path, index=False)

print("Result:", out_path)



Result: C:\datum-api-examples-main\OriON\CRACEN\work\corr_beta_pairs.parquet


In [29]:
import os, time, gc
import pandas as pd
import numpy as np
from typing import Optional

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception:
    pa = pq = None


def _load_corrbeta_mapping_parquet(path: str) -> pd.DataFrame:
    """
    Читає parquet з мапою:
      A) ['x_ticker','y_ticker','corr','beta'] (API-форма), або
      B) ['ticker','bench','corr','beta'] (вже нормалізована).
    Повертає DF: ['ticker','bench','corr','beta'] (UPPER, без дублікатів; при дублях
    лишаємо рядок із найбільшою corr).
    """
    if pq is None:
        raise RuntimeError("Потрібен pyarrow для читання parquet.")

    df = pq.read_table(path).to_pandas()
    inv = {c.strip().lower(): c for c in df.columns}

    has_api = ("x_ticker" in inv) and ("y_ticker" in inv)
    has_direct = ("ticker" in inv) and ("bench" in inv)
    if not (has_api or has_direct):
        available = ", ".join(df.columns.astype(str).tolist())
        raise KeyError(
            "Mapping parquet повинен містити або ['x_ticker','y_ticker','corr','beta'], "
            "або ['ticker','bench','corr','beta'].\n"
            f"У файлі наявні колонки: {available}"
        )

    if has_api:
        df = df.rename(columns={inv["x_ticker"]: "ticker", inv["y_ticker"]: "bench"})
    else:
        df = df.rename(columns={inv["ticker"]: "ticker", inv["bench"]: "bench"})

    for c in ("corr", "beta"):
        if c in inv:
            df = df.rename(columns={inv[c]: c})
        else:
            df[c] = pd.NA

    df = df[["ticker", "bench", "corr", "beta"]].copy()
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["bench"] = df["bench"].astype(str).str.strip().str.upper()
    df["corr"] = pd.to_numeric(df["corr"], errors="coerce")
    df["beta"] = pd.to_numeric(df["beta"], errors="coerce")

    return (df.sort_values(["ticker", "corr"], ascending=[True, False])
              .drop_duplicates(subset=["ticker"], keep="first")
              .reset_index(drop=True))


def attach_bench_corr_beta_parquet(
    input_parquet_path: str,
    corrbeta_parquet_path: str,
    out_parquet_path: str,
    *,
    batch_rows: int = 2_000_000,
    log_every_batches: int = 5,
    round_decimals: Optional[int] = 4,
    insert_before: str = "pct_vs_prev_close",
    compression: str = "snappy",
) -> str:
    """
    Потоково читає великий Parquet (інтрадеї), додає 'bench','corr','beta' по 'ticker'.

    Замість merge на кожен батч (+ strip/upper над мільйонами рядків) нормалізація і
    пошук у мапі йдуть по унікальних тікерах, а результат розгортається за кодами.
    """
    if pq is None or pa is None:
        raise RuntimeError("Потрібен pyarrow (pa, pq) для роботи з Parquet.")

    t0 = time.time()
    mapping = _load_corrbeta_mapping_parquet(corrbeta_parquet_path)
    if round_decimals is not None:
        mapping["corr"] = mapping["corr"].round(round_decimals)
        mapping["beta"] = mapping["beta"].round(round_decimals)

    if mapping.empty:
        raise ValueError(
            f"Мапа bench/corr/beta порожня: {corrbeta_parquet_path}. "
            "Крок побудови x_to_best_etf_by_lvl3 не дав жодного рядка — "
            "інакше bench/corr/beta були б null у всіх рядках."
        )

    m_bench = dict(zip(mapping["ticker"], mapping["bench"]))
    m_corr = dict(zip(mapping["ticker"], mapping["corr"]))
    m_beta = dict(zip(mapping["ticker"], mapping["beta"]))

    if os.path.exists(out_parquet_path):
        os.remove(out_parquet_path)

    pf = pq.ParquetFile(input_parquet_path)
    writer = None
    total_rows = 0

    print(f"▶️ START merge → {out_parquet_path}", flush=True)

    for i, batch in enumerate(pf.iter_batches(batch_size=batch_rows), 1):
        df = batch.to_pandas()

        if "ticker" not in df.columns:
            raise KeyError("У вхідному parquet відсутній стовпець 'ticker'.")

        codes, cats = _ticker_codes(df["ticker"])
        df["ticker"] = _expand(np.asarray(cats.to_numpy(), dtype=object), codes, None)
        # dtype="string" тримає bench рядковим і в батчі без жодного збігу: інакше
        # суцільний None дає pyarrow-тип null і схема розійдеться з першим батчем
        df["bench"] = pd.array(_map_by_ticker(codes, cats, m_bench, numeric=False), dtype="string")
        df["corr"] = _map_by_ticker(codes, cats, m_corr, numeric=True)
        df["beta"] = _map_by_ticker(codes, cats, m_beta, numeric=True)

        cols = [c for c in df.columns if c not in ("bench", "corr", "beta")]
        if insert_before in cols:
            idx = cols.index(insert_before)
            df = df[cols[:idx] + ["bench", "corr", "beta"] + cols[idx:]]
        else:
            df = df[cols + ["bench", "corr", "beta"]]

        table = pa.Table.from_pandas(df, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out_parquet_path, table.schema, compression=compression)
        elif not table.schema.equals(writer.schema, check_metadata=False):
            table = table.cast(writer.schema)
        writer.write_table(table)

        total_rows += len(df)
        if i % log_every_batches == 0:
            print(f"[batch {i:>4}] rows={total_rows:,}  elapsed={time.time()-t0:,.1f}s", flush=True)

    if writer is not None:
        writer.close()

    print(f"✅ DONE rows={total_rows:,} → {out_parquet_path}  in {time.time()-t0:,.1f}s", flush=True)
    return out_parquet_path


# опціональний аліас під старе ім'я (щоб існуючий код не міняти)
def attach_bench_corr_beta_file(**kwargs) -> str:
    return attach_bench_corr_beta_parquet(**kwargs)


In [30]:
# ── ЗЛИТО В ОДИН ПРОХІД ─────────────────────────────────────────────
# Цей крок тепер робить build_final_in_one_pass у комірці 46: весь файл
# читається й пишеться ОДИН раз замість п'яти. Старий виклик лишено
# закоментованим — щоб відкотити, розкоментуй його і прибери виклик у 46.
# out_path = attach_bench_corr_beta_parquet(
#     input_parquet_path=str(WORK_DIR / "premarket_rows_enriched_with_pct.parquet"),
#     corrbeta_parquet_path=str(WORK_DIR / "corr_beta_pairs.parquet"),
#     out_parquet_path=str(WORK_DIR / "premarket_rows_final.parquet"),
#     batch_rows=2_000_000,
#     round_decimals=4,
#     insert_before="pct_vs_prev_close",
#     compression="snappy",
# )
# print("Готово:", out_path)


In [31]:
import os
import pandas as pd

def peek_parquet(path: str, n: int = 20, columns=None):
    """
    Легкий перегляд великого Parquet:
      - друкує шлях, кількість рядків (із метаданих), кількість колонок і їх назви
      - потоково читає і показує перші n рядків (optionally лише вказані columns)
      - повертає DataFrame із прев’ю (n рядків)
    Вимагає pyarrow.
    """
    try:
        import pyarrow.parquet as pq
    except Exception as e:
        raise RuntimeError(
            "Потрібен пакет 'pyarrow' для читання parquet. Встанови: pip install pyarrow"
        ) from e

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    pf = pq.ParquetFile(path)
    total_rows = pf.metadata.num_rows if pf.metadata is not None else None
    schema_cols = [f.name for f in pf.schema_arrow]
    print(f"Файл: {path}")
    if total_rows is not None:
        print(f"Рядків (meta): {total_rows:,}")
    print(f"Колонок: {len(schema_cols)}")
    print("Схема:", ", ".join(schema_cols))

    # Збираємо перші n рядків потоково
    need = n
    parts = []
    for batch in pf.iter_batches(batch_size=min(100_000, max(1, n)), columns=columns):
        df_part = batch.to_pandas()
        parts.append(df_part)
        need -= len(df_part)
        if need <= 0:
            break

    head_df = pd.concat(parts, ignore_index=True).head(n) if parts else pd.DataFrame()
    # Коротка діагностика по ключових нових стовпцях (якщо є)
    for c in ["bench", "corr", "beta", "pct_vs_prev_close"]:
        if c in head_df.columns:
            na = int(head_df[c].isna().sum())
            print(f"NaN у '{c}' (у прев'ю): {na}/{len(head_df)}")

    print("\nПерші рядки:")
    with pd.option_context("display.max_rows", n, "display.max_columns", 200, "display.width", 200):
        print(head_df)

    return head_df


In [32]:
# ── ЗЛИТО В ОДИН ПРОХІД ─────────────────────────────────────────────
# Проміжного premarket_rows_final.parquet більше не існує: bench/corr/beta
# додаються за той самий прохід, що й решта. Готовий файл — у комірці 47.


In [33]:
# список тикерів
tickers = [
    'ARKK', 'COPX', 'DRAM', 'FCX', 'FXI', 'GDX', 'IBIT', 'IGV', 'ITA', 'IWM',
    'KRE', 'KWEB', 'NASA', 'QQQ', 'SLV', 'SOXX', 'SPY', 'UNG', 'URA', 'XBI',
    'XLB', 'XLE', 'XLF', 'XLP', 'XLU', 'XLV', 'XOP', 'XRT',
]

# створення DataFrame
df_bench = pd.DataFrame(tickers, columns=['ticker'])


In [34]:
keep = ["ticker","dt","o","h","l","c","v"]

errs = fetch_intraday_v3_for_all_tickers(
    tickers_df=df_bench,
    start= start,
    end= end,
    interval=1,
    chunk_days=30,
    parallel_chunks=True,
    max_workers_chunks=12,
    parallel_tickers=True,
    max_workers_tickers=12,
    save_every=50,
    out_dir=str(ETF_DIR),
    out_prefix="intraday",
    gzip=True,
    keep_columns=keep,
    verbose=True,
    include_blue_ocean=True,
)

# Збої API більше не маскуються під «порожні дні»: усе, що тут перелічено, НЕ позначено
# готовим. Запускай клітинку повторно, доки errs не стане порожнім.
if errs is not None and len(errs):
    print(f"⚠️ {len(errs)} тікерів зі збоями — перезапусти клітинку:")
    print(errs.head(20).to_string())


▶ Починаємо: 28 тікерів (пропущено 0 по старому маніфесту) | 2026-08-01 00:00:00 → 2026-11-01 23:59:59 | interval=1 | next batch #0163 | run=93dc5f8d


   workers: tickers=12, chunks=12 | blue_ocean=True | day-manifest=True


   day-manifest: повністю готових тікерів пропущено = 0


   📉 пропущені дні (в межах window): 2026-08-01 → 2026-10-31 (92 днів)


   📈 обробляються дні: 2026-11-01 → 2026-11-01 (1 днів)


   psutil · старт · 🧠 RAM: 769.9 MB


... (4 outputs left out)

💾 batch 163 → 1,997 рядків (всього 1,997 у цьому запуску) → C:\datum-api-examples-main\OriON\CRACEN\work\ETF\intraday_0163.csv.gz


   ✅ після flush · 🧠 RAM: 773.4 MB


   🔚 фінал · 🧠 RAM: 773.4 MB


🏁 Готово. Нових рядків: 1,997 → C:\datum-api-examples-main\OriON\CRACEN\work\ETF | Час: 00:01


In [35]:
# Один прохід по ETF-батчах: premarket-рядки + денні open/close.
_, df_oc = collect_premarket_and_oc(
    folder=str(ETF_DIR),
    out_path=str(WORK_DIR / "premarket_etf_rows.parquet"),
    oc_out_path=str(WORK_DIR / "oc_etf.parquet"),
    start_time="00:00:00",
    end_time="23:59:00",
    build_oc=True,
    max_workers=4,
)


▶️ START: C:\datum-api-examples-main\OriON\CRACEN\work\ETF/intraday_*.csv.gz → C:\datum-api-examples-main\OriON\CRACEN\work\premarket_etf_rows.parquet  files=163 workers=4


  [1/163] intraday_0004.csv.gz  in=1,760 out=1,760 elapsed=0.0s


  [2/163] intraday_0003.csv.gz  in=10,518 out=10,518 elapsed=0.0s


  [3/163] intraday_0005.csv.gz  in=21,091 out=21,091 elapsed=0.1s


  [4/163] intraday_0002.csv.gz  in=31,264 out=31,264 elapsed=0.1s


  [5/163] intraday_0006.csv.gz  in=41,355 out=41,355 elapsed=0.1s


... (155 outputs left out)

  [161/163] intraday_0163.csv.gz  in=4,761,683 out=4,761,683 elapsed=3.2s

  [162/163] intraday_0161.csv.gz  in=4,781,878 out=4,781,878 elapsed=3.2s


  [163/163] intraday_0162.csv.gz  in=4,800,368 out=4,800,368 elapsed=3.2s


✅ DONE  in=4,800,368 → out=4,800,368  to C:\datum-api-examples-main\OriON\CRACEN\work\premarket_etf_rows.parquet  in 3.4s

In [36]:
# oc_etf.parquet уже побудовано в клітинці 34 (той самий прохід по файлах).
_oc_etf_path = str(WORK_DIR / "oc_etf.parquet")
if os.path.exists(_oc_etf_path):
    df_oc = pd.read_parquet(_oc_etf_path)
else:
    df_oc = build_oc_from_intraday_folder(folder=str(ETF_DIR), out_path=_oc_etf_path)
print(df_oc.head())


  ticker        date   open  close
0   ARKK  2026-05-01  75.85  76.84
1   ARKK  2026-05-04  77.24  78.21
2   ARKK  2026-05-05  78.40  76.62
3   ARKK  2026-05-06  77.33  79.54
4   ARKK  2026-05-07  79.40  78.09


In [37]:
# Ті самі оптимізовані функції, що й у клітинці 22 — тут лише інші дефолти обгортки
# (ETF-файли замість акційних).
def merge_intraday_and_oc_in_folder(
    folder: str = "CRACEN",
    intraday_filename: str = "premarket_etf_rows.parquet",
    oc_filename: str = "oc_etf.parquet",
    out_filename: str = "premarket_etf_rows_enriched.parquet",
    batch_rows: int = 2_000_000,
    log_every_batches: int = 5,
) -> str:
    intraday_path = os.path.join(folder, intraday_filename)
    oc_path = os.path.join(folder, oc_filename)
    out_path = os.path.join(folder, out_filename)

    if not os.path.exists(intraday_path):
        raise FileNotFoundError(f"Не знайдено інтрадей: {intraday_path}")
    if not os.path.exists(oc_path):
        raise FileNotFoundError(f"Не знайдено OC-файл: {oc_path}")
    os.makedirs(folder, exist_ok=True)

    merge_intraday_and_oc_to_file(
        intraday_parquet_path=intraday_path,
        oc_parquet_path=oc_path,
        out_path=out_path,
        batch_rows=batch_rows,
        log_every_batches=log_every_batches,
    )
    return out_path


In [38]:
out_path = merge_intraday_and_oc_in_folder(folder=str(WORK_DIR))
print("Результат у файлі:", out_path)


▶️ START: merge → C:\datum-api-examples-main\OriON\CRACEN\work\premarket_etf_rows_enriched.parquet | batch_rows≈2,000,000


✅ DONE  rows=4,800,368  -> C:\datum-api-examples-main\OriON\CRACEN\work\premarket_etf_rows_enriched.parquet  in 9.7s


Результат у файлі: C:\datum-api-examples-main\OriON\CRACEN\work\premarket_etf_rows_enriched.parquet


In [39]:
# Та сама оптимізована add_pct_vs_prev_close_parquet із клітинки 24 —
# тут лише інші дефолти обгортки (ETF-файли, колонка Bench%).
def add_pct_vs_prev_close_in_folder(
    folder: str = "CRACEN",
    input_filename: str = "premarket_etf_rows_enriched.parquet",
    output_filename: str = "premarket_etf_rows_enriched_with_pct.parquet",
    batch_rows: int = 2_000_000,
    round_to: Optional[int] = 2,
    log_every: int = 5,
    value_col: str = "o",                 # ← open vs prev_close
    out_col: str = "Bench%",              # назва нової колонки
) -> str:
    """Обгортка під твою папку CRACEN."""
    os.makedirs(folder, exist_ok=True)
    src = os.path.join(folder, input_filename)
    dst = os.path.join(folder, output_filename)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Не знайдено вхідний файл: {src}")
    return add_pct_vs_prev_close_parquet(
        input_path=src,
        output_path=dst,
        batch_rows=batch_rows,
        round_to=round_to,
        log_every=log_every,
        value_col=value_col,
        out_col=out_col,
    )


In [40]:
out_path = add_pct_vs_prev_close_in_folder(folder=str(WORK_DIR))
print("Готово:", out_path)


▶️ START  file=C:\datum-api-examples-main\OriON\CRACEN\work\premarket_etf_rows_enriched.parquet  →  C:\datum-api-examples-main\OriON\CRACEN\work\premarket_etf_rows_enriched_with_pct.parquet  batch_rows≈2,000,000  using=o


✅ DONE  rows=4,800,368  →  C:\datum-api-examples-main\OriON\CRACEN\work\premarket_etf_rows_enriched_with_pct.parquet  in 4.7s


Готово: C:\datum-api-examples-main\OriON\CRACEN\work\premarket_etf_rows_enriched_with_pct.parquet


In [41]:
import os, gc, time
import pandas as pd
from typing import Optional

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception:
    pa = pq = None


def merge_bench_pct_from_etf(
    base_path: str = "CRACEN/work/premarket_rows_enriched_with_pct.parquet",
    etf_path:  str = "CRACEN/work/premarket_etf_rows_enriched_with_pct.parquet",
    out_path:  str = "CRACEN/work/premarket_rows_with_bench_pct.parquet",
    *,
    batch_rows: int = 2_000_000,
    align_to_minute: bool = False,        # якщо в секундах відрізняється — підрівняти до хвилини
    round_decimals: Optional[int] = 2,    # округлити Bench% до 2 знаків
    insert_before: Optional[str] = None,  # наприклад, "pct_vs_prev_close"; інакше в кінець
    compression: str = "snappy",
    log_every_batches: int = 5,
) -> str:
    """
    Мерджить Bench% із ETF-файла у базовий по ключах ('bench'↔'ticker', 'dt').

    Прискорення: ETF-мапа читається одним pq.read_table (лише 3 потрібні колонки)
    замість iter_batches + concat + gc; ключ 'dt' лишається рядком —
    лексикографічний порядок 'YYYY-MM-DD HH:MM:SS' збігається з хронологічним,
    тож per-row парсинг дат не потрібен (крім align_to_minute=True).
    """
    if pq is None or pa is None:
        raise RuntimeError("Потрібен pyarrow: pip install pyarrow")

    if not os.path.exists(base_path):
        raise FileNotFoundError(base_path)
    if not os.path.exists(etf_path):
        raise FileNotFoundError(etf_path)

    t0 = time.time()

    # ---------- 1) Мапа з ETF-файла: (bench, dt) -> Bench% ----------
    etf_names = [f.name for f in pq.ParquetFile(etf_path).schema_arrow]
    cl = {c.lower().strip(): c for c in etf_names}
    for col in ("ticker", "dt"):
        if col not in cl:
            raise KeyError(f"У ETF-файлі відсутня колонка '{col}'")
    val_col = cl.get("bench%") or cl.get("pct_vs_prev_close")
    if val_col is None:
        raise KeyError("У ETF-файлі не знайдено 'bench%' або 'pct_vs_prev_close'.")

    etf_map = pq.read_table(etf_path, columns=[cl["ticker"], cl["dt"], val_col]).to_pandas()
    etf_map.columns = ["bench", "dt", "Bench%"]
    etf_map["bench"] = etf_map["bench"].astype(str).str.strip().str.upper()
    etf_map["Bench%"] = pd.to_numeric(etf_map["Bench%"], errors="coerce")
    if round_decimals is not None:
        etf_map["Bench%"] = etf_map["Bench%"].round(round_decimals)

    _dt_is_str = (etf_map["dt"].dtype == object) or pd.api.types.is_string_dtype(etf_map["dt"])
    if align_to_minute or not _dt_is_str:
        etf_map["dt"] = pd.to_datetime(etf_map["dt"], errors="coerce")
        if align_to_minute:
            etf_map["dt"] = etf_map["dt"].dt.floor("T")
    else:
        etf_map["dt"] = etf_map["dt"].astype(str)

    etf_map = (etf_map.dropna(subset=["dt"])
                      .sort_values(["bench", "dt"], kind="stable")
                      .drop_duplicates(subset=["bench", "dt"], keep="last")
                      .reset_index(drop=True))

    # ---------- 2) Потоково проходимо базовий файл ----------
    if os.path.exists(out_path):
        os.remove(out_path)

    pf_base = pq.ParquetFile(base_path)
    writer = None
    total_rows = 0

    print(f"▶️ START merge → {out_path}", flush=True)
    for i, batch in enumerate(pf_base.iter_batches(batch_size=batch_rows), 1):
        df = batch.to_pandas()

        for col in ("bench", "dt"):
            if col not in df.columns:
                raise KeyError(f"У базовому файлі відсутня колонка '{col}'")

        codes, cats = _ticker_codes(df["bench"])
        df["bench"] = _expand(np.asarray(cats.to_numpy(), dtype=object), codes, None)

        _base_dt_is_str = (df["dt"].dtype == object) or pd.api.types.is_string_dtype(df["dt"])
        if align_to_minute or not _base_dt_is_str:
            df["dt"] = pd.to_datetime(df["dt"], errors="coerce")
            if align_to_minute:
                df["dt"] = df["dt"].dt.floor("T")
        else:
            df["dt"] = df["dt"].astype(str)

        merged = df.merge(etf_map, on=["bench", "dt"], how="left")

        if "Bench%" in merged.columns and insert_before and insert_before in merged.columns:
            cols = [c for c in merged.columns if c != "Bench%"]
            pos = cols.index(insert_before)
            merged = merged[cols[:pos] + ["Bench%"] + cols[pos:]]

        table = pa.Table.from_pandas(merged, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema, compression=compression)
        writer.write_table(table)

        total_rows += len(merged)
        if i % log_every_batches == 0:
            print(f"[batch {i:>4}] rows={total_rows:,}  elapsed={time.time()-t0:,.1f}s", flush=True)

    if writer is not None:
        writer.close()

    print(f"✅ DONE rows={total_rows:,} → {out_path}  in {time.time()-t0:,.1f}s", flush=True)
    return out_path


In [42]:
# ── ЗЛИТО В ОДИН ПРОХІД ─────────────────────────────────────────────
# Цей крок тепер робить build_final_in_one_pass у комірці 46: весь файл
# читається й пишеться ОДИН раз замість п'яти. Старий виклик лишено
# закоментованим — щоб відкотити, розкоментуй його і прибери виклик у 46.
# merge_bench_pct_from_etf(
#     base_path=str(WORK_DIR / "premarket_rows_final.parquet"),
#     etf_path =str(WORK_DIR / "premarket_etf_rows_enriched_with_pct.parquet"),
#     out_path =str(WORK_DIR / "premarket_rows_with_bench_pct.parquet"),
#     batch_rows=2_000_000,
#     align_to_minute=False,       # вмикай, якщо секунди/мілісекунди «пливуть»
#     round_decimals=2,
#     insert_before=None,          # або, наприклад, "pct_vs_prev_close"
#     compression="snappy",
# )


In [43]:
import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


def _pick_first_existing(df: pd.DataFrame, candidates):
    cols_l = {c.lower().strip(): c for c in df.columns}
    for c in candidates:
        if c.lower() in cols_l:
            return cols_l[c.lower()]
    return None


_REPORT_FAILS = {"n": 0}
_GAP_FAILS = {"n": 0}


def _datum_request_retry(endpoint, params, *, max_retries=6, backoff=0.5):
    """Повторює запит при 429/мережевих збоях з наростаючою паузою.
    Вичерпавши спроби — кидає останній виняток."""
    import random, time
    for attempt in range(max_retries + 1):
        try:
            return DatumApi.data_request(endpoint, params)
        except Exception:
            if attempt == max_retries:
                raise
            time.sleep(backoff * (2 ** attempt) * (0.9 + 0.2 * random.random()))


def get_reports(ticker, last_n_reports, start_date, end_date):
    try:
        params = {
            "ticker": str(ticker),
            "start_move_date": str(start_date)[:10],
            "end_move_date": str(end_date)[:10],
        }
        reports_df = _datum_request_retry("/reports", params)
        if reports_df is None or reports_df.empty:
            return pd.DataFrame(columns=["ticker", "report_date"])

        date_col = _pick_first_existing(
            reports_df,
            ["report_date", "move_date", "date", "dt", "datetime", "published_at", "created_at"],
        )
        if date_col is None:
            return pd.DataFrame(columns=["ticker", "report_date"])

        out = reports_df.copy()
        out["report_date"] = pd.to_datetime(out[date_col], errors="coerce").dt.strftime("%Y-%m-%d")
        out["ticker"] = str(ticker).upper()
        out = out.dropna(subset=["report_date"])
        out = out.sort_values("report_date").tail(int(last_n_reports))
        return out[["ticker", "report_date"]].reset_index(drop=True)
    except Exception as e:
        _REPORT_FAILS["n"] += 1
        print(f"Report fetch failed for {ticker}: {e}")
        return pd.DataFrame(columns=["ticker", "report_date"])


def get_reports_for_tickers(tickers_df, start_date, end_date, last_n_reports=20,
                            max_workers=8, max_fail_ratio=0.15):
    # 16 потоків без пауз давали суцільні 429; з retry вистачає меншого числа
    tickers = tickers_df["ticker"].dropna().astype(str).str.upper().unique().tolist()
    parts = []
    _REPORT_FAILS["n"] = 0

    # Ті самі репорти вже завантажила комірка 11. Беремо їх лише коли вікно,
    # last_n_reports і набір тикерів збігаються; інакше — звичайний фетч.
    _c = globals().get("CRACEN_FETCH_CACHE") or {}
    if (_c.get("window") == (str(start_date)[:10], str(end_date)[:10])
            and _c.get("last_n_reports") == int(last_n_reports)
            and set(tickers) <= _c.get("tickers", set())):
        _r = _c["reports"].rename(columns={"date": "report_date"})
        _r = (_r[_r["ticker"].isin(tickers)]
                .drop_duplicates(["ticker", "report_date"]).reset_index(drop=True))
        print(f"Reports: узято з кешу комірки 11 ({len(_r)} рядків), фетч пропущено")
        return _r, _r.groupby("ticker")["report_date"].apply(set).to_dict()

    def fetch(t):
        return get_reports(ticker=t, last_n_reports=last_n_reports, start_date=start_date, end_date=end_date)

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(fetch, t): t for t in tickers}
        for f in tqdm(as_completed(futures), total=len(futures), desc="Reports"):
            rep = f.result()
            if rep is not None and not rep.empty:
                parts.append(rep)

    if tickers and _REPORT_FAILS["n"] / len(tickers) > max_fail_ratio:
        raise RuntimeError(
            f"API відмовив на {_REPORT_FAILS['n']} з {len(tickers)} тикерів "
            f"({_REPORT_FAILS['n'] / len(tickers) * 100:.1f}% > {max_fail_ratio * 100:.0f}%). "
            "Репорти вийшли б неповними."
        )

    if not parts:
        return pd.DataFrame(columns=["ticker", "report_date"]), {}

    reports = pd.concat(parts, ignore_index=True).drop_duplicates(["ticker", "report_date"])
    report_days_map = reports.groupby("ticker")["report_date"].apply(set).to_dict()
    return reports, report_days_map


def get_gaps_for_tickers_excluding_reports(tickers_df, start_date, end_date, report_days_map,
                                           max_workers=8, max_fail_ratio=0.15):
    tickers = tickers_df["ticker"].dropna().astype(str).str.upper().unique().tolist()
    _GAP_FAILS["n"] = 0
    _GAP_FAILS["empty"] = 0

    # Кеш комірки 11 беремо для КОЖНОГО тикера, що в ньому є, а докачуємо лише відсутні. Раніше кеш
    # діяв за умови set(tickers) <= cache: один зайвий тикер викидав увесь кеш, і ~7000 тикерів
    # перекачувались наново; порожні відповіді API при цьому мовчки губились (fetch -> None без
    # лічильника збою) - так sigma лишилась лише у ~1200 тикерів (алфавітний хвіст STC..ZYME).
    _c = globals().get("CRACEN_FETCH_CACHE") or {}
    cached_part = None
    to_fetch = tickers
    if _c.get("window") == (str(start_date)[:10], str(end_date)[:10]) and "gaps" in _c:
        _have = set(_c.get("tickers", set()))
        _use = [t for t in tickers if t in _have]
        _g = _c["gaps"]
        _g = _g[_g["ticker"].isin(_use)].copy()
        _rep = pd.DataFrame([(t, d) for t, ds in (report_days_map or {}).items() for d in ds],
                            columns=["ticker", "date"])
        if len(_rep):
            _g = _g.merge(_rep.assign(_r=1), on=["ticker", "date"], how="left")
            _g = _g[_g["_r"].isna()].drop(columns="_r")
        cached_part = _g.reset_index(drop=True)
        to_fetch = [t for t in tickers if t not in _have]
        print(f"Gaps: з кешу комірки 11 {len(_use)} тикерів ({len(cached_part)} рядків), докачуємо {len(to_fetch)}")
        if not to_fetch:
            return cached_part

    def fetch(t):
        params = {
            "ticker": t,
            "start_date": str(start_date)[:10],
            "end_date": str(end_date)[:10],
            "format": "json_records",
        }
        try:
            df_gap = _datum_request_retry("/daily/gaps", params)
            if df_gap is None or df_gap.empty:
                # порожня відповідь під лімітом виглядає так само, як тикер без гепів - перепитуємо раз
                import time
                time.sleep(1.0)
                df_gap = _datum_request_retry("/daily/gaps", params)
            if df_gap is None or df_gap.empty:
                _GAP_FAILS["empty"] = _GAP_FAILS.get("empty", 0) + 1
                return None

            date_col = _pick_first_existing(df_gap, ["date", "move_date", "dt", "datetime", "day"])
            if date_col is None:
                return None

            out = df_gap.copy()
            out["ticker"] = out["ticker"].astype(str).str.upper() if "ticker" in out.columns else t
            out["date"] = pd.to_datetime(out[date_col], errors="coerce").dt.strftime("%Y-%m-%d")
            out["gap"] = pd.to_numeric(out.get("gap"), errors="coerce")
            out = out.dropna(subset=["date", "gap"])

            report_days = report_days_map.get(t, set())
            if report_days:
                out = out[~out["date"].isin(report_days)]

            return out if not out.empty else None
        except Exception as e:
            _GAP_FAILS["n"] += 1
            print(f"Gap fetch failed for {t}: {e}")
            return None

    all_results = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(fetch, t): t for t in to_fetch}
        for f in tqdm(as_completed(futures), total=len(futures), desc="Gaps (ex-reports)"):
            result = f.result()
            if result is not None:
                all_results.append(result)

    _bad = _GAP_FAILS["n"] + _GAP_FAILS.get("empty", 0)
    if to_fetch and _bad / len(to_fetch) > max_fail_ratio:
        raise RuntimeError(
            f"API відмовив/повернув порожньо на {_bad} з {len(to_fetch)} тикерів "
            f"({_bad / len(to_fetch) * 100:.1f}% > {max_fail_ratio * 100:.0f}%). "
            "Гепи вийшли б неповними."
        )

    if cached_part is not None:
        all_results.append(cached_part)
    if all_results:
        return pd.concat(all_results, ignore_index=True)
    return pd.DataFrame(columns=["ticker", "date", "gap"])


def add_rolling_sigma_cap(
    df,
    value_col,
    window=20,
    group_col="ticker",
    date_col="date",
    cap_q=0.8,
    min_periods=None,
):
    """Rolling std по попередньому вікну з обрізанням хвостів на квантилі cap_q.

    Раніше це був rolling(...).apply(python_func) — окремий виклик Python на кожне
    вікно кожного тікера. Тепер вікна будуються через sliding_window_view і квантиль/
    clip/std рахуються одразу по всій матриці вікон групи (чисто numpy).
    Результат збігається з попередньою реалізацією до 1e-12.
    """
    if min_periods is None:
        min_periods = window

    out = df.sort_values([group_col, date_col], kind="stable").copy()
    col = f"{value_col}_sigma_{window}d_cap"

    # Рідкісний випадок min_periods != window має іншу семантику (часткові вікна) —
    # там лишаємо стару, повільну, але точну реалізацію.
    if min_periods != window:
        def compute_sigma(s):
            x = s.shift(1)

            def f(arr):
                arr = arr[~np.isnan(arr)]
                if len(arr) < min_periods:
                    return np.nan
                cap = np.quantile(np.abs(arr), cap_q)
                arr_capped = np.clip(arr, -cap, cap)
                return np.std(arr_capped, ddof=1)

            return x.rolling(window, min_periods=min_periods).apply(f, raw=True)

        out[col] = out.groupby(group_col)[value_col].transform(compute_sigma)
        return out

    vals = pd.to_numeric(out[value_col], errors="coerce").to_numpy(dtype="float64")
    codes, _ = pd.factorize(out[group_col], sort=False)
    res = np.full(vals.shape, np.nan)

    order = np.argsort(codes, kind="stable")
    bounds = np.flatnonzero(np.diff(codes[order])) + 1
    for sl in np.split(order, bounds):
        n = sl.size
        if n <= window:
            continue
        x = vals[sl]
        # вікно з ПОПЕРЕДНІХ значень (еквівалент shift(1) у старій версії)
        w = np.lib.stride_tricks.sliding_window_view(x[:-1], window)   # (n-window, window)
        cap = np.quantile(np.abs(w), cap_q, axis=1, keepdims=True)
        res[sl[window:]] = np.clip(w, -cap, cap).std(axis=1, ddof=1)

    out[col] = res
    return out


# 1) Report days in the working window
reports_df, report_days_map = get_reports_for_tickers(
    TICKERS,
    start_date=start_date_str,
    end_date=end_date_str,
    last_n_reports=20,
)
print("Report rows:", len(reports_df))

# 2) Gaps without report days
df_gaps_all = get_gaps_for_tickers_excluding_reports(
    TICKERS,
    start_date=start_date_str,
    end_date=end_date_str,
    report_days_map=report_days_map,
)
print("Rows in gaps (without report days):", len(df_gaps_all))

# 3) Rolling sigma
sigma_daily = add_rolling_sigma_cap(
    df=df_gaps_all,
    value_col="gap",
    window=20,
    group_col="ticker",
    date_col="date",
    cap_q=0.8,
    min_periods=20,
)

sigma_col = "gap_sigma_20d_cap"

std_by_ticker = (
    sigma_daily.dropna(subset=[sigma_col])
    .sort_values(["ticker", "date"])
    .groupby("ticker", as_index=False)
    .tail(1)[["ticker", sigma_col]]
    .rename(columns={sigma_col: "sigma"})
    .reset_index(drop=True)
)

# 4) Добір sigma для тикерів, що випали: гепи з oc.parquet (open / попередній close - те саме, з чого
# рахується Stack%), ті самі дні репортів, те саме вікно і та сама формула.
_all_tk = set(TICKERS["ticker"].dropna().astype(str).str.upper())
_missing = sorted(_all_tk - set(std_by_ticker["ticker"]))
print(f"Sigma з API-гепів: {len(std_by_ticker)} з {len(_all_tk)} тикерів, бракує {len(_missing)}")
if _missing:
    _oc = pd.read_parquet(WORK_DIR / "oc.parquet", columns=["ticker", "date", "open", "close"])
    _oc["ticker"] = _oc["ticker"].astype(str).str.upper()
    _oc["date"] = pd.to_datetime(_oc["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    _oc = _oc.dropna(subset=["date"]).sort_values(["ticker", "date"], kind="stable")
    _oc["gap"] = (pd.to_numeric(_oc["open"], errors="coerce")
                  / pd.to_numeric(_oc.groupby("ticker")["close"].shift(1), errors="coerce") - 1.0) * 100.0
    _fb = _oc[_oc["ticker"].isin(_missing)
              & (_oc["date"] >= str(start_date_str)[:10]) & (_oc["date"] <= str(end_date_str)[:10])]
    _fb = _fb.replace([np.inf, -np.inf], np.nan).dropna(subset=["gap"])
    _rep_fb = pd.DataFrame([(t, d) for t, ds in report_days_map.items() for d in ds], columns=["ticker", "date"])
    if len(_rep_fb):
        _fb = _fb.merge(_rep_fb.assign(_r=1), on=["ticker", "date"], how="left")
        _fb = _fb[_fb["_r"].isna()].drop(columns="_r")
    _fb_sigma = add_rolling_sigma_cap(_fb[["ticker", "date", "gap"]], value_col="gap", window=20,
                                      group_col="ticker", date_col="date", cap_q=0.8, min_periods=20)
    _fb_std = (_fb_sigma.dropna(subset=[sigma_col]).sort_values(["ticker", "date"])
               .groupby("ticker", as_index=False).tail(1)[["ticker", sigma_col]]
               .rename(columns={sigma_col: "sigma"}))
    std_by_ticker = pd.concat([std_by_ticker, _fb_std], ignore_index=True).drop_duplicates("ticker", keep="first")
    print(f"Добрано з oc.parquet: {len(_fb_std)}")

_cov = len(std_by_ticker) / max(1, len(_all_tk))
print(f"Tickers with sigma: {len(std_by_ticker)} з {len(_all_tk)} ({_cov:.1%})")
if _cov < 0.85:
    raise RuntimeError(f"sigma є лише у {_cov:.1%} тикерів - гепи вийшли неповними, final.parquet не збирати")
std_by_ticker.head()

Reports: узято з кешу комірки 11 (3996 рядків), фетч пропущено
Report rows: 3996


Gaps: з кешу комірки 11 7098 тикерів (442577 рядків), докачуємо 0
Rows in gaps (without report days): 442577


Sigma з API-гепів: 7022 з 7098 тикерів, бракує 76


Добрано з oc.parquet: 0
Tickers with sigma: 7022 з 7098 (98.9%)


,ticker,sigma
0,A,0.512934
1,AA,0.945077
2,AAAU,0.772638
3,AACG,3.803630
4,AADX,0.763126


In [44]:
std_by_ticker



,ticker,sigma
0,A,0.512934
1,AA,0.945077
2,AAAU,0.772638
3,AACG,3.803630
4,AADX,0.763126
...,...,...
7017,ZVOL,0.431505
7018,ZVRA,0.459633
7019,ZWS,0.766665
7020,ZYBT,2.130289


In [45]:
import os, time, gc
import pandas as pd
import numpy as np
from typing import Optional

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception:
    pa = pq = None


def _prepare_sigma_df(
    sigma_df: Optional[pd.DataFrame] = None,
    sigma_path: Optional[str] = None
) -> pd.DataFrame:
    """
    Готує мапу ['ticker','sigma'] з DataFrame або з файлу (.parquet/.csv).
    Нормалізує регістр, типи і прибирає дублікати по 'ticker'.
    """
    if sigma_df is None and sigma_path is None:
        raise ValueError("Передай або sigma_df, або sigma_path.")

    if sigma_df is None:
        ext = os.path.splitext(sigma_path)[1].lower()
        if ext == ".parquet":
            sigma_df = pd.read_parquet(sigma_path)
        else:
            sigma_df = pd.read_csv(sigma_path, low_memory=False)

    df = sigma_df.copy()
    cols_lower = {c.lower().strip(): c for c in df.columns}
    tcol = cols_lower.get("ticker")
    scol = cols_lower.get("sigma")
    if tcol is None or scol is None:
        raise KeyError("Очікую колонки 'ticker' і 'sigma' у даних зі скріна.")

    df = df[[tcol, scol]].rename(columns={tcol: "ticker", scol: "sigma"})
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["sigma"] = pd.to_numeric(df["sigma"], errors="coerce")
    df = (df.sort_values(["ticker"])
            .drop_duplicates(subset=["ticker"], keep="last")
            .reset_index(drop=True))
    return df[["ticker", "sigma"]]


def add_sigma_by_ticker_to_parquet(
    base_path: str = "CRACEN/work/premarket_rows_with_bench_pct.parquet",
    out_path:  Optional[str] = None,
    *,
    sigma_df: Optional[pd.DataFrame] = None,   # напр. твій std_by_ticker
    sigma_path: Optional[str] = None,          # або шлях до файла з цими даними
    batch_rows: int = 2_000_000,
    compression: str = "snappy",
    insert_before: Optional[str] = None,       # напр. "Bench%"; якщо None — в кінець
    log_every_batches: int = 5,
    round_decimals: Optional[int] = 6,
) -> str:
    """
    Потоково додає колонку 'sigma' (за 'ticker') до великого Parquet.

    Як і в клітинці 28: нормалізація тікера і пошук у мапі йдуть по унікальних
    значеннях, а не по кожному з мільйонів рядків; per-batch merge не потрібен.
    """
    if pq is None or pa is None:
        raise RuntimeError("Потрібен pyarrow (pyarrow, pyarrow.parquet).")

    if not os.path.exists(base_path):
        raise FileNotFoundError(base_path)

    if out_path is None:
        folder, fname = os.path.split(base_path)
        name, ext = os.path.splitext(fname)
        out_path = os.path.join(folder, f"{name}_with_sigma{ext or '.parquet'}")

    sigma_map = _prepare_sigma_df(sigma_df=sigma_df, sigma_path=sigma_path)
    if round_decimals is not None:
        sigma_map["sigma"] = sigma_map["sigma"].round(round_decimals)
    sd = dict(zip(sigma_map["ticker"], sigma_map["sigma"]))

    tmp_path = out_path + f".tmp_{os.getpid()}_{int(time.time())}"
    pf = pq.ParquetFile(base_path)
    writer = None
    total = 0
    t0 = time.time()

    print(f"▶️ START merge sigma → {out_path}", flush=True)
    for i, batch in enumerate(pf.iter_batches(batch_size=batch_rows), 1):
        df = batch.to_pandas()
        if "ticker" not in df.columns:
            raise KeyError("У базовому файлі відсутня колонка 'ticker'.")

        codes, cats = _ticker_codes(df["ticker"])
        df["ticker"] = _expand(np.asarray(cats.to_numpy(), dtype=object), codes, None)
        df["sigma"] = _map_by_ticker(codes, cats, sd, numeric=True)

        cols = [c for c in df.columns if c != "sigma"]
        if insert_before and insert_before in cols:
            idx = cols.index(insert_before)
            df = df[cols[:idx] + ["sigma"] + cols[idx:]]
        else:
            df = df[cols + ["sigma"]]

        table = pa.Table.from_pandas(df, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(tmp_path, table.schema, compression=compression)
        writer.write_table(table)

        total += len(df)
        if i % log_every_batches == 0:
            print(f"[batch {i:>4}] rows={total:,}  elapsed={time.time()-t0:,.1f}s", flush=True)

    if writer is not None:
        writer.close()

    if os.path.exists(out_path):
        os.remove(out_path)
    os.replace(tmp_path, out_path)

    print(f"✅ DONE rows={total:,} → {out_path}  in {time.time()-t0:,.1f}s", flush=True)
    return out_path


In [46]:
def build_final_in_one_pass(
    intraday_parquet_path: str,
    oc_parquet_path: str,
    corrbeta_parquet_path: str,
    etf_parquet_path: str,
    out_parquet_path: str,
    *,
    sigma_df=None,
    sigma_path=None,
    batch_rows: int = 2_000_000,
    round_decimals=4,
    stack_round_to=None,
    bench_round_decimals=2,
    compression: str = "snappy",
    log_every_batches: int = 5,
) -> str:
    """Один потоковий прохід замість п'яти: oc-мердж → Stack% → bench/corr/beta
    → Bench% → sigma. Замінює merge_intraday_and_oc_to_file +
    add_pct_vs_prev_close_parquet + attach_bench_corr_beta_parquet +
    merge_bench_pct_from_etf + add_sigma_by_ticker_to_parquet.

    Результат побайтово збігається з ланцюжком; економія — у тому, що ~3 ГБ
    читаються й пишуться один раз, а не шість.
    """
    t0 = time.time()

    # ---------- side-входи: готуються ОДИН раз ----------
    oc_prepared = _prep_oc_prev_close(pq.read_table(oc_parquet_path).to_pandas())

    mapping = _load_corrbeta_mapping_parquet(corrbeta_parquet_path)
    if mapping.empty:
        raise ValueError(
            f"Мапа bench/corr/beta порожня: {corrbeta_parquet_path}. "
            "Крок побудови x_to_best_etf_by_lvl3 не дав жодного рядка."
        )
    if round_decimals is not None:
        mapping["corr"] = mapping["corr"].round(round_decimals)
        mapping["beta"] = mapping["beta"].round(round_decimals)
    m_bench = dict(zip(mapping["ticker"], mapping["bench"]))
    m_corr = dict(zip(mapping["ticker"], mapping["corr"]))
    m_beta = dict(zip(mapping["ticker"], mapping["beta"]))

    etf_names = [f.name for f in pq.ParquetFile(etf_parquet_path).schema_arrow]
    cl = {c.lower().strip(): c for c in etf_names}
    val_col = cl.get("bench%") or cl.get("pct_vs_prev_close")
    etf_map = pq.read_table(etf_parquet_path,
                            columns=[cl["ticker"], cl["dt"], val_col]).to_pandas()
    etf_map.columns = ["bench", "dt", "Bench%"]
    etf_map["bench"] = etf_map["bench"].astype(str).str.strip().str.upper()
    etf_map["Bench%"] = pd.to_numeric(etf_map["Bench%"], errors="coerce")
    if bench_round_decimals is not None:
        etf_map["Bench%"] = etf_map["Bench%"].round(bench_round_decimals)
    etf_map["dt"] = etf_map["dt"].astype(str)
    etf_map = (etf_map.dropna(subset=["dt"])
                      .sort_values(["bench", "dt"], kind="stable")
                      .drop_duplicates(subset=["bench", "dt"], keep="last")
                      .reset_index(drop=True))

    sig = _prepare_sigma_df(sigma_df=sigma_df, sigma_path=sigma_path)
    m_sigma = dict(zip(sig["ticker"], sig["sigma"]))

    if os.path.exists(out_parquet_path):
        os.remove(out_parquet_path)

    ORDER = ["ticker", "dt", "o", "h", "l", "c", "v", "date", "open", "prev_close",
             "Stack%", "bench", "corr", "beta", "Bench%", "sigma"]

    pf = pq.ParquetFile(intraday_parquet_path)
    writer = None
    total_rows = 0
    print(f"▶️ START one-pass → {out_parquet_path}", flush=True)

    for i, batch in enumerate(pf.iter_batches(batch_size=batch_rows), 1):
        df = batch.to_pandas()

        # 1) день + open/prev_close (ключ — сирий ticker, як у ланцюжку)
        df = df[df["dt"].notna()]
        df = df.assign(date=_date_from_dt(df["dt"]))
        df = df.merge(oc_prepared, on=["ticker", "date"], how="left")
        df["prev_close"] = _rebase_prev_close_after_session_close(df)

        # 2) Stack%
        num = pd.to_numeric(df["o"], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
        den = pd.to_numeric(df["prev_close"], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
        with np.errstate(divide="ignore", invalid="ignore"):
            pct = np.where(den == 0, np.nan, num / den - 1.0) * 100.0
        if stack_round_to is not None:
            pct = np.round(pct, int(stack_round_to))
        df["Stack%"] = pct

        # 3) bench/corr/beta (тут ticker нормалізується — як у ланцюжку)
        codes, cats = _ticker_codes(df["ticker"])
        df["ticker"] = _expand(np.asarray(cats.to_numpy(), dtype=object), codes, None)
        df["bench"] = _map_by_ticker(codes, cats, m_bench, numeric=False)
        df["corr"] = _map_by_ticker(codes, cats, m_corr, numeric=True)
        df["beta"] = _map_by_ticker(codes, cats, m_beta, numeric=True)

        # 4) sigma — теж по codes/cats, тому рахуємо ДО мерджа
        df["sigma"] = _map_by_ticker(codes, cats, m_sigma, numeric=True)

        # 5) Bench% по (bench, dt); bench лишається object для мерджа
        df["dt"] = df["dt"].astype(str)
        df = df.merge(etf_map, on=["bench", "dt"], how="left")

        # string-dtype один раз наприкінці: тримає тип bench стабільним між батчами
        df["bench"] = pd.array(df["bench"], dtype="string")

        df = df[[c for c in ORDER if c in df.columns]]
        table = pa.Table.from_pandas(df, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out_parquet_path, table.schema, compression=compression)
        elif not table.schema.equals(writer.schema, check_metadata=False):
            table = table.cast(writer.schema)
        writer.write_table(table)

        total_rows += len(df)
        if i % log_every_batches == 0:
            print(f"[batch {i:>4}] rows={total_rows:,} elapsed={time.time()-t0:,.1f}s", flush=True)
        del df, table, batch
        gc.collect()

    if writer is not None:
        writer.close()
    print(f"✅ DONE rows={total_rows:,} → {out_parquet_path} in {time.time()-t0:,.1f}s", flush=True)
    return out_parquet_path


In [47]:
# std_by_ticker columns: ['ticker', 'sigma']
# Один прохід замість п'яти: oc-мердж → Stack% → bench/corr/beta → Bench% → sigma.
# Замінює комірки 23, 25, 29, 41 і колишній add_sigma_by_ticker_to_parquet.
sigma_out_path = build_final_in_one_pass(
    intraday_parquet_path=str(WORK_DIR / "premarket_rows.parquet"),
    oc_parquet_path=str(WORK_DIR / "oc.parquet"),
    corrbeta_parquet_path=str(WORK_DIR / "corr_beta_pairs.parquet"),
    etf_parquet_path=str(WORK_DIR / "premarket_etf_rows_enriched_with_pct.parquet"),
    out_parquet_path=str(WORK_DIR / "premarket_rows_with_bench_pct_with_sigma.parquet"),
    sigma_df=std_by_ticker,
    batch_rows=2_000_000,
    round_decimals=4,
    bench_round_decimals=2,
    compression="snappy",
)

# Validation: output parquet must contain sigma matching std_by_ticker
check_df = pd.read_parquet(sigma_out_path, columns=["ticker", "sigma"])
if "sigma" not in check_df.columns:
    raise KeyError("Column 'sigma' is missing in parquet after merge")

file_sigma = (
    check_df.dropna(subset=["ticker"])
    .assign(ticker=lambda d: d["ticker"].astype(str).str.upper().str.strip())
    .drop_duplicates(subset=["ticker"], keep="last")
    .set_index("ticker")["sigma"]
)

exp_sigma = (
    std_by_ticker.dropna(subset=["ticker"])
    .assign(ticker=lambda d: d["ticker"].astype(str).str.upper().str.strip())
    .drop_duplicates(subset=["ticker"], keep="last")
    .set_index("ticker")["sigma"]
)

common = file_sigma.index.intersection(exp_sigma.index)
if len(common) == 0:
    raise ValueError("No common tickers to validate sigma between std_by_ticker and parquet")

diff = (file_sigma.loc[common] - exp_sigma.loc[common]).abs()
max_diff = float(diff.max()) if len(diff) else 0.0
print("sigma_out_path:", sigma_out_path)
print("tickers in std_by_ticker:", len(exp_sigma))
print("tickers checked in parquet:", len(common))
print("max |file_sigma - expected_sigma|:", max_diff)



▶️ START one-pass → C:\datum-api-examples-main\OriON\CRACEN\work\premarket_rows_with_bench_pct_with_sigma.parquet


[batch    5] rows=10,000,000 elapsed=46.6s


[batch   10] rows=20,000,000 elapsed=84.0s


[batch   15] rows=30,000,000 elapsed=122.1s


[batch   20] rows=40,000,000 elapsed=159.2s


[batch   25] rows=50,000,000 elapsed=196.6s


... (4 outputs left out)

[batch   50] rows=100,000,000 elapsed=384.6s


[batch   55] rows=110,000,000 elapsed=422.4s


✅ DONE rows=111,611,206 → C:\datum-api-examples-main\OriON\CRACEN\work\premarket_rows_with_bench_pct_with_sigma.parquet in 428.7s


sigma_out_path: C:\datum-api-examples-main\OriON\CRACEN\work\premarket_rows_with_bench_pct_with_sigma.parquet
tickers in std_by_ticker: 7022
tickers checked in parquet: 7020
max |file_sigma - expected_sigma|: 0.0


In [48]:
peek_parquet(str(WORK_DIR / "premarket_rows_with_bench_pct_with_sigma.parquet"), n=21150)



Файл: C:\datum-api-examples-main\OriON\CRACEN\work\premarket_rows_with_bench_pct_with_sigma.parquet
Рядків (meta): 111,611,206
Колонок: 16
Схема: ticker, dt, o, h, l, c, v, date, open, prev_close, Stack%, bench, corr, beta, Bench%, sigma
NaN у 'bench' (у прев'ю): 468/21150
NaN у 'corr' (у прев'ю): 468/21150
NaN у 'beta' (у прев'ю): 468/21150

Перші рядки:


      ticker                   dt           o           h           l           c          v        date    open  prev_close        Stack% bench    corr    beta  Bench%     sigma
0          A  2026-08-03 08:59:00  137.399994  137.899994  137.399994  137.899994      200.0  2026-08-03  140.29      138.37 -7.010234e-01   XLV  0.5065  0.3844     NaN  0.512934
1          A  2026-08-03 09:00:00  138.399994  138.399994  138.399994  138.399994      173.0  2026-08-03  140.29      138.37  2.167659e-02   XLV  0.5065  0.3844     NaN  0.512934
2          A  2026-08-03 09:30:00  140.289993  140.289993  137.130005  139.360001    25052.0  2026-08-03  140.29      138.37  1.387579e+00   XLV  0.5065  0.3844    0.66  0.512934
3          A  2026-08-03 09:31:00  139.960007  139.960007  139.464996  139.464996     2500.0  2026-08-03  140.29      138.37  1.149098e+00   XLV  0.5065  0.3844    0.97  0.512934
4          A  2026-08-03 09:32:00  139.100006  139.970001  139.100006  139.464996     7500.0  2026-08-03 

,ticker,dt,o,h,l,c,v,date,open,prev_close,Stack%,bench,corr,beta,Bench%,sigma
0,A,2026-08-03 08:59:00,137.399994,137.899994,137.399994,137.899994,200.0,2026-08-03,140.29,138.37,-7.010234e-01,XLV,0.5065,0.3844,NaN,0.512934
1,A,2026-08-03 09:00:00,138.399994,138.399994,138.399994,138.399994,173.0,2026-08-03,140.29,138.37,2.167659e-02,XLV,0.5065,0.3844,NaN,0.512934
2,A,2026-08-03 09:30:00,140.289993,140.289993,137.130005,139.360001,25052.0,2026-08-03,140.29,138.37,1.387579e+00,XLV,0.5065,0.3844,0.66,0.512934
3,A,2026-08-03 09:31:00,139.960007,139.960007,139.464996,139.464996,2500.0,2026-08-03,140.29,138.37,1.149098e+00,XLV,0.5065,0.3844,0.97,0.512934
4,A,2026-08-03 09:32:00,139.100006,139.970001,139.100006,139.464996,7500.0,2026-08-03,140.29,138.37,5.275754e-01,XLV,0.5065,0.3844,1.12,0.512934
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21145,ACI,2026-08-03 11:30:00,11.605000,11.608900,11.590000,11.590000,11029.0,2026-08-03,11.59,11.58,2.158855e-01,XLP,0.3835,0.2942,0.08,0.660958
21146,ACI,2026-08-03 11:31:00,11.595000,11.595000,11.580000,11.585000,7365.0,2026-08-03,11.59,11.58,1.295360e-01,XLP,0.3835,0.2942,-0.02,0.660958
21147,ACI,2026-08-03 11:32:00,11.585000,11.590000,11.580000,11.590000,4152.0,2026-08-03,11.59,11.58,4.317822e-02,XLP,0.3835,0.2942,-0.01,0.660958
21148,ACI,2026-08-03 11:33:00,11.585000,11.590000,11.580000,11.590000,2957.0,2026-08-03,11.59,11.58,4.317822e-02,XLP,0.3835,0.2942,-0.04,0.660958


In [49]:
import os, time, gc
import pandas as pd
import numpy as np
from typing import Optional, List

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    import pyarrow.dataset as ds
    import pyarrow.compute as pc
except Exception:
    pa = pq = ds = pc = None


def _as_f64(col):
    """ChunkedArray -> float64 без походу в pandas (fallback лише для нечислових типів)."""
    if pa.types.is_floating(col.type) or pa.types.is_integer(col.type):
        return col.cast(pa.float64()) if col.type != pa.float64() else col
    if pa.types.is_decimal(col.type):
        return col.cast(pa.float64())
    return pa.chunked_array([pa.array(
        pd.to_numeric(col.to_pandas(), errors="coerce").to_numpy(dtype="float64"))])


def _round_like_numpy(col, ndigits):
    """np.round над Arrow-колонкою + NaN -> null (як робив pa.Table.from_pandas)."""
    if len(col) == 0:
        return col
    v = col.to_numpy(zero_copy_only=False)   # може бути read-only zero-copy view
    if ndigits is not None:
        v = np.round(v, int(ndigits))
    return pa.array(v, from_pandas=True)


def _add_tp_deviation_devsig(table, *, round_tp_dev=2, round_devsig=6, dt_as_timestamp=False):
    """t_p / deviation / dev_sig як арифметика Arrow над колонками таблиці.

    Раніше кожен батч ходив у pandas і назад (batch.to_pandas() +
    pa.Table.from_pandas()). Для рядкових колонок ticker/dt/date/bench це означало
    матеріалізацію мільйонів python-об'єктів str на кожному батчі й повне
    перезбирання Arrow-масивів — найдорожча частина того циклу. Тут таблиця
    лишається в Arrow від читання до запису.
    """
    names = {n.strip().lower(): n for n in table.column_names}
    stack_col = names.get("stack%") or names.get("pct_vs_prev_close")
    bench_col = names.get("bench%")
    beta_col = names.get("beta")
    sigma_col = names.get("sigma")

    need = {
        "stack%/pct_vs_prev_close": stack_col,
        "bench%": bench_col,
        "beta": beta_col,
        "sigma": sigma_col,
    }
    missing = [k for k, v in need.items() if v is None]
    if missing:
        raise KeyError(f"У вхідному файлі бракує колонок: {missing}")

    drop = [c for c in ("t_p", "deviation", "dev_sig") if c in table.column_names]
    if drop:
        table = table.drop_columns(drop) if hasattr(table, "drop_columns") else table.drop(drop)

    stack = _as_f64(table[stack_col])
    beta = _as_f64(table[beta_col])
    bench = _as_f64(table[bench_col])
    sigma = _as_f64(table[sigma_col])

    tp = pc.multiply(stack, beta)
    dev = pc.subtract(tp, bench)
    denom = pc.if_else(pc.equal(sigma, 0), pa.scalar(None, pa.float64()), sigma)  # sigma 0 -> NaN
    dsg = pc.divide(dev, denom)

    # Округлення саме через np.round, а не pc.round: на значеннях, які вже округлені
    # вище по пайплайну (Stack% до 2 знаків, beta до 4), добуток лягає впритул до
    # межі .005, і pc.round подекуди дає результат на 1 ULP інший за np.round.
    # np.round тут ще й трохи швидший, а pa.array(..., from_pandas=True) відтворює
    # ту саму семантику NaN -> null, що давав старий pa.Table.from_pandas().
    tp = _round_like_numpy(tp, round_tp_dev)
    dev = _round_like_numpy(dev, round_tp_dev)
    dsg = _round_like_numpy(dsg, round_devsig)

    if dt_as_timestamp and "dt" in table.column_names and pa.types.is_string(table["dt"].type):
        i = table.column_names.index("dt")
        table = table.set_column(i, "dt", pc.strptime(table["dt"], format="%Y-%m-%d %H:%M:%S",
                                                      unit="ns", error_is_null=True))

    return (table.append_column("t_p", tp)
                 .append_column("deviation", dev)
                 .append_column("dev_sig", dsg))


def _drop_dup_ticker_dt(table):
    """Прибирає дублі (ticker, dt) у вже ВІДСОРТОВАНІЙ за (ticker, date, dt) таблиці.

    Дублі виникали, бо dedup у fetch працює лише в межах одного батча, а той самий
    (ticker, dt) лежав у кількох батчах (докачка / перекриття діапазонів): у final було
    ~8.1 млн ідентичних рядків у 2 479 тікерів (NVDA мав 780 хв замість 390 за сесію).
    Дублі ідентичні (перевірено на o/h/l/c/v), тому лишаємо перший.
    Повертає (table, кількість_видалених)."""
    n = table.num_rows
    if n < 2:
        return table, 0
    tk = table["ticker"].combine_chunks()
    dt = table["dt"].combine_chunks()
    same = pc.and_(pc.equal(tk.slice(1), tk.slice(0, n - 1)),
                   pc.equal(dt.slice(1), dt.slice(0, n - 1)))
    same = pc.fill_null(same, False)
    dropped = pc.sum(same.cast(pa.int64())).as_py() or 0
    if not dropped:
        return table, 0
    keep = pa.concat_arrays([pa.array([True]), pc.invert(same)])
    return table.filter(keep), int(dropped)


def _sort_keys(schema_names):
    keys = [("ticker", "ascending")]
    if "date" in schema_names:
        keys.append(("date", "ascending"))
    if "dt" in schema_names:
        keys.append(("dt", "ascending"))
    return keys


def recompute_tp_deviation_devsig(
    input_path: str = "CRACEN/work/premarket_rows_with_bench_pct_with_sigma.parquet",
    output_path: str = "CRACEN/work/premarket_rows_with_bench_pct_with_sigma_v2.parquet",
    chunksize: int = 2_000_000,   # batch_rows для потокового режиму / бакетування
    round_tp_dev: Optional[int] = 2,
    round_devsig: Optional[int] = 6,
    compression: str = "snappy",
    *,
    sort_final_by_ticker: bool = True,
    max_rows_in_memory: int = 20_000_000,
    dt_as_timestamp: bool = False,
    row_group_size: int = 2_000_000,
) -> str:
    """
    Перерахунок:
      - t_p       = stack% * beta
      - deviation = t_p - bench%
      - dev_sig   = deviation / sigma  (0 -> NaN)
    Стирає старі t_p / deviation / dev_sig, якщо були.

    Якщо sort_final_by_ticker=True, output_path фізично відсортований:
    ticker A→Z, всередині тикера date,dt.

    Що змінилось проти попередньої версії:
      1. Обчислення і сортування — ОДИН прохід. Раніше спершу писався повний
         tmp_unsorted, потім він читався назад і переписувався відсортованим:
         3 читання + 3 записи всього файла. Тепер 1 читання + 1 запис
         (або 2+2, якщо файл не влазить у пам'ять і потрібні бакети).
      2. Уся арифметика в Arrow — без batch.to_pandas()/pa.Table.from_pandas()
         на кожному батчі (це матеріалізувало рядкові колонки як python-об'єкти).
      3. Сортування ПЕРЕД додаванням t_p/deviation/dev_sig — take() переставляє
         на 3 float64-колонки менше.
      4. Бакетування по тикерах — через pc.index_in замість pandas .map по рядках.
      5. row_group_size=2M замість дефолтного 1M: менше row-груп → швидший запис,
         трохи менший файл, і стратегії-споживачі (вони читають
         iter_batches(batch_size=1_000_000)) теж читають його швидше.
    """
    if pq is None or pa is None or ds is None:
        raise RuntimeError("Потрібен pyarrow (pyarrow, pyarrow.parquet, pyarrow.dataset).")

    if not os.path.exists(input_path):
        raise FileNotFoundError(input_path)

    t0 = time.time()
    dataset = ds.dataset(input_path, format="parquet")
    schema_names = [f.name for f in dataset.schema]
    total_rows = dataset.count_rows()

    tmp_out = output_path + f".tmp_{os.getpid()}_{int(time.time())}"
    if os.path.exists(tmp_out):
        os.remove(tmp_out)

    def _finish(rows, mode):
        if os.path.exists(output_path):
            os.remove(output_path)
        os.replace(tmp_out, output_path)
        print(f"✅ DONE ({mode}): {rows:,} rows → {output_path} | "
              f"total_time={time.time()-t0:,.1f}s", flush=True)
        return output_path

    # ---------- 1) без сортування: чистий стрім, один прохід ----------
    # (УВАГА: у цьому режимі дублі (ticker, dt) не видаляються — для final лишай sort_final_by_ticker=True)
    if not sort_final_by_ticker:
        print(f"▶ START (stream): {input_path} → {output_path} | "
              f"rows={total_rows:,} batch={chunksize:,}", flush=True)
        writer = None
        written = 0
        try:
            for bi, rb in enumerate(dataset.to_batches(batch_size=chunksize), 1):
                if rb.num_rows == 0:
                    continue
                tbl = _add_tp_deviation_devsig(
                    pa.Table.from_batches([rb]), round_tp_dev=round_tp_dev,
                    round_devsig=round_devsig, dt_as_timestamp=dt_as_timestamp)
                if writer is None:
                    writer = pq.ParquetWriter(tmp_out, tbl.schema, compression=compression)
                writer.write_table(tbl, row_group_size=row_group_size)
                written += tbl.num_rows
                if bi % 5 == 0:
                    print(f"[batch {bi:>4}] rows={written:,}  "
                          f"elapsed={time.time()-t0:,.1f}s", flush=True)
        finally:
            if writer is not None:
                writer.close()
        return _finish(written, "stream")

    keys = _sort_keys(schema_names)
    if "ticker" not in schema_names:
        raise KeyError("У вхідному parquet немає колонки 'ticker' — неможливо відсортувати по тикерах.")

    # ---------- 2) влазить у пам'ять: читання → сорт → рахунок → запис ----------
    if total_rows <= max_rows_in_memory:
        print(f"▶ START (in-memory sort): {input_path} → {output_path} | "
              f"rows={total_rows:,}", flush=True)
        table = dataset.to_table().sort_by(keys)
        table, n_dup = _drop_dup_ticker_dt(table)
        if n_dup:
            print(f"🧹 видалено дублів (ticker, dt): {n_dup:,}", flush=True)
        table = _add_tp_deviation_devsig(
            table, round_tp_dev=round_tp_dev, round_devsig=round_devsig,
            dt_as_timestamp=dt_as_timestamp)
        pq.write_table(table, tmp_out, compression=compression,
                       row_group_size=row_group_size)
        written = table.num_rows
        del table
        gc.collect()
        return _finish(written, "in-memory sort")

    # ---------- 3) не влазить: один прохід у бакети, далі сорт кожного ----------
    n_buckets = int(np.ceil(total_rows / float(max_rows_in_memory)))
    tickers = pc.unique(dataset.to_table(columns=["ticker"]).column("ticker")).to_pylist()
    tickers = sorted(str(t) for t in tickers if t is not None)
    # суміжні діапазони тикерів → порядок бакетів = порядок тикерів
    edges = np.array_split(np.arange(len(tickers)), n_buckets)
    rank_to_bucket = np.zeros(len(tickers), dtype="int32")
    for b, idxs in enumerate(edges):
        rank_to_bucket[idxs] = b
    ticker_set = pa.array(tickers, type=pa.string())
    rank_to_bucket_arr = pa.array(rank_to_bucket)

    print(f"▶ START (bucketed sort): {input_path} → {output_path} | rows={total_rows:,} "
          f"tickers={len(tickers):,} buckets={n_buckets}", flush=True)

    bucket_paths = [tmp_out + f".b{b}" for b in range(n_buckets)]
    bwriters: List[Optional["pq.ParquetWriter"]] = [None] * n_buckets
    try:
        for rb in dataset.to_batches(batch_size=chunksize):
            if rb.num_rows == 0:
                continue
            tbl = pa.Table.from_batches([rb])
            # ранг тикера в відсортованому списку → номер бакета, повністю в Arrow
            ranks = pc.index_in(tbl.column("ticker").cast(pa.string()), value_set=ticker_set)
            bidx = pc.take(rank_to_bucket_arr, ranks)
            for b in range(n_buckets):
                mask = pc.equal(bidx, b)
                if not pc.any(mask).as_py():
                    continue
                part = tbl.filter(mask)
                if part.num_rows == 0:
                    continue
                if bwriters[b] is None:
                    bwriters[b] = pq.ParquetWriter(bucket_paths[b], part.schema,
                                                   compression=compression)
                bwriters[b].write_table(part)
    finally:
        for w in bwriters:
            if w is not None:
                w.close()

    writer = None
    written = 0
    dup_total = 0
    try:
        for b, bp in enumerate(bucket_paths):
            if not os.path.exists(bp):
                continue
            tbl = pq.read_table(bp).sort_by(keys)
            tbl, n_dup = _drop_dup_ticker_dt(tbl)
            dup_total += n_dup
            tbl = _add_tp_deviation_devsig(
                tbl, round_tp_dev=round_tp_dev, round_devsig=round_devsig,
                dt_as_timestamp=dt_as_timestamp)
            if writer is None:
                writer = pq.ParquetWriter(tmp_out, tbl.schema, compression=compression)
            writer.write_table(tbl, row_group_size=row_group_size)
            written += tbl.num_rows
            del tbl
            gc.collect()
            print(f"  bucket {b+1}/{n_buckets}  written={written:,}  dup_dropped={dup_total:,}  "
                  f"elapsed={time.time()-t0:,.1f}s", flush=True)
    finally:
        if writer is not None:
            writer.close()
        for bp in bucket_paths:
            if os.path.exists(bp):
                os.remove(bp)

    return _finish(written, "bucketed sort")


def _write_final_sorted_alphabetical(
    input_path: str,
    output_path: str,
    *,
    compression: str = "snappy",
    max_rows_in_memory: int = 20_000_000,
    log_every: int = 200,            # лишено для сумісності виклику
    roundtrip_pandas: bool = True,   # лишено для сумісності виклику
):
    """Лише сортування parquet: ticker A→Z, всередині тикера (date, dt).

    Обчислення t_p/deviation/dev_sig тепер робить сам recompute_tp_deviation_devsig
    у тому ж проході, тож окремо ця функція потрібна лише якщо треба відсортувати
    вже готовий файл.
    """
    if ds is None or pq is None or pa is None:
        raise RuntimeError("Потрібен pyarrow.dataset (pyarrow>=6).")

    dataset = ds.dataset(input_path, format="parquet")
    schema_names = [f.name for f in dataset.schema]
    if "ticker" not in schema_names:
        raise KeyError("У вхідному parquet немає колонки 'ticker' — неможливо відсортувати по тикерах.")
    keys = _sort_keys(schema_names)

    total_rows = dataset.count_rows()
    t0 = time.time()
    tmp_out = output_path + f".tmp_sorted_{os.getpid()}_{int(time.time())}"
    if os.path.exists(tmp_out):
        os.remove(tmp_out)

    if total_rows <= max_rows_in_memory:
        print(f"▶ SORT (in-memory): rows={total_rows:,}", flush=True)
        table = dataset.to_table().sort_by(keys)
        pq.write_table(table, tmp_out, compression=compression)
        written = table.num_rows
        del table
        gc.collect()
    else:
        n_buckets = int(np.ceil(total_rows / float(max_rows_in_memory)))
        tickers = pc.unique(dataset.to_table(columns=["ticker"]).column("ticker")).to_pylist()
        tickers = sorted(str(t) for t in tickers if t is not None)
        rank_to_bucket = np.zeros(len(tickers), dtype="int32")
        for b, idxs in enumerate(np.array_split(np.arange(len(tickers)), n_buckets)):
            rank_to_bucket[idxs] = b
        ticker_set = pa.array(tickers, type=pa.string())
        rank_to_bucket_arr = pa.array(rank_to_bucket)
        print(f"▶ SORT (bucketed): rows={total_rows:,} tickers={len(tickers):,} "
              f"buckets={n_buckets}", flush=True)

        bucket_paths = [tmp_out + f".b{b}" for b in range(n_buckets)]
        bwriters: List[Optional["pq.ParquetWriter"]] = [None] * n_buckets
        try:
            for rb in dataset.to_batches():
                if rb.num_rows == 0:
                    continue
                tbl = pa.Table.from_batches([rb])
                ranks = pc.index_in(tbl.column("ticker").cast(pa.string()), value_set=ticker_set)
                bidx = pc.take(rank_to_bucket_arr, ranks)
                for b in range(n_buckets):
                    mask = pc.equal(bidx, b)
                    if not pc.any(mask).as_py():
                        continue
                    part = tbl.filter(mask)
                    if part.num_rows == 0:
                        continue
                    if bwriters[b] is None:
                        bwriters[b] = pq.ParquetWriter(bucket_paths[b], part.schema,
                                                       compression=compression)
                    bwriters[b].write_table(part)
        finally:
            for w in bwriters:
                if w is not None:
                    w.close()

        writer = None
        written = 0
        try:
            for b, bp in enumerate(bucket_paths):
                if not os.path.exists(bp):
                    continue
                tbl = pq.read_table(bp).sort_by(keys)
                if writer is None:
                    writer = pq.ParquetWriter(tmp_out, tbl.schema, compression=compression)
                writer.write_table(tbl)
                written += tbl.num_rows
                del tbl
                gc.collect()
        finally:
            if writer is not None:
                writer.close()
            for bp in bucket_paths:
                if os.path.exists(bp):
                    os.remove(bp)

    if os.path.exists(output_path):
        os.remove(output_path)
    os.replace(tmp_out, output_path)
    print(f"✅ SORT DONE: {written:,} rows → {output_path} | time={time.time()-t0:,.1f}s", flush=True)
    return output_path


In [50]:
recompute_tp_deviation_devsig(
    input_path=str(WORK_DIR / "premarket_rows_with_bench_pct_with_sigma.parquet"),
    output_path=str(FINAL_PATH),
    chunksize=2_000_000,
    round_tp_dev=2,
    round_devsig=6,
)



▶ START (bucketed sort): C:\datum-api-examples-main\OriON\CRACEN\work\premarket_rows_with_bench_pct_with_sigma.parquet → C:\datum-api-examples-main\OriON\CRACEN\final.parquet | rows=111,611,206 tickers=7,812 buckets=6


  bucket 1/6  written=18,669,279  dup_dropped=0  elapsed=110.0s


  bucket 2/6  written=36,711,782  dup_dropped=0  elapsed=132.4s


  bucket 3/6  written=54,226,009  dup_dropped=0  elapsed=155.3s


  bucket 4/6  written=72,869,107  dup_dropped=0  elapsed=178.6s


  bucket 5/6  written=92,905,133  dup_dropped=0  elapsed=204.6s


  bucket 6/6  written=111,611,206  dup_dropped=0  elapsed=228.1s


✅ DONE (bucketed sort): 111,611,206 rows → C:\datum-api-examples-main\OriON\CRACEN\final.parquet | total_time=228.3s


'C:\\datum-api-examples-main\\OriON\\CRACEN\\final.parquet'

In [51]:
# ================== АУДИТ final.parquet: повнота днів / хвилин / o-h-l-c ==================
# Перевіряє ВСЕ, а не вибірково:
#   1) o,h,l,c,v не null; h >= max(o,c) та l <= min(o,c) (з допуском на float32);
#   2) жодних дублів (ticker, dt);
#   3) для «щільних» тікерів (>= min_rth_rows хв у сесії 09:30–15:59 у типовий день): будь-який
#      торговий день МІЖ їхнім першим і останнім днем, коли ринок торгується, а цей тікер — ні (саме так виглядав NVDA/AAPL без 10–28 серпня), та дні з неповною сесією;
#   4) кількість тікерів на день різко (<85% медіани сусідніх днів) просіла → підозра на дірку фетчу.
# Файл розріджений за задумом (бар є лише для хвилин, де були угоди), тому «хвилина без бару» для
# неліквідних тікерів — не помилка; повноту хвилин міряємо на щільних тікерах.
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

def audit_final_coverage(final_path, *, rth_from="09:30", rth_to="15:59", min_rth_rows=380,
                         alive_share=0.2, drop_ratio=0.85, raise_on_fail=False):
    pf = pq.ParquetFile(final_path)
    need = ["ticker", "dt", "o", "h", "l", "c", "v"]
    miss = [c for c in need if c not in pf.schema_arrow.names]
    if miss:
        raise KeyError(f"У final немає колонок: {miss}")
    n_rows = n_dup = n_null = n_bad = 0
    prev = None
    parts = []
    for gi in range(pf.num_row_groups):
        t = pf.read_row_group(gi, columns=need)
        n = t.num_rows
        if n == 0:
            continue
        n_rows += n
        for c in "ohlcv":
            n_null += t[c].null_count
        o, h, l, c_ = (t[x].to_numpy() .astype("float64") for x in "ohlc")
        with np.errstate(invalid="ignore"):
            n_bad += int(np.sum((h < np.maximum(o, c_) - 1e-3) | (l > np.minimum(o, c_) + 1e-3)))
        tk = t["ticker"].combine_chunks()
        dt = t["dt"].combine_chunks()
        if not pa.types.is_string(dt.type):
            dt = pc.strftime(dt, format="%Y-%m-%d %H:%M:%S")
        same = pc.fill_null(pc.and_(pc.equal(tk.slice(1), tk.slice(0, n - 1)),
                                    pc.equal(dt.slice(1), dt.slice(0, n - 1))), False)
        n_dup += pc.sum(same.cast(pa.int64())).as_py() or 0
        if prev is not None and prev == (tk[0].as_py(), dt[0].as_py()):
            n_dup += 1
        prev = (tk[n - 1].as_py(), dt[n - 1].as_py())
        hm = pc.utf8_slice_codeunits(dt, 11, 16)
        rth = pc.and_(pc.greater_equal(hm, rth_from), pc.less_equal(hm, rth_to)).cast(pa.int64())
        g = (pa.table({"ticker": tk, "date": pc.utf8_slice_codeunits(dt, 0, 10), "rth": rth, "rows": rth})
               .group_by(["ticker", "date"]).aggregate([("rth", "sum"), ("rows", "count")]))
        parts.append(g.to_pandas().rename(columns={"rth_sum": "rth", "rows_count": "rows"}))
    a = pd.concat(parts, ignore_index=True).groupby(["ticker", "date"], as_index=False).sum()
    a["wd"] = pd.to_datetime(a["date"]).dt.weekday

    problems = []
    print(f"рядків: {n_rows:,} | null у o/h/l/c/v: {n_null:,} | порушень OHLC: {n_bad:,} | дублів (ticker,dt): {n_dup:,}")
    if n_null:
        problems.append("null у OHLCV")
    if n_bad:
        problems.append("h/l не охоплюють o/c")
    if n_dup:
        problems.append("дублі (ticker,dt)")

    wk = a[a["wd"] < 5]
    days = sorted(wk["date"].unique())
    per_day = wk.groupby("date")["ticker"].nunique().reindex(days)
    sus = []
    for i, d in enumerate(days):
        nb = [per_day.iloc[j] for j in range(max(0, i - 10), min(len(days), i + 11)) if j != i]
        if nb and per_day.iloc[i] < drop_ratio * float(np.median(nb)):
            sus.append((d, int(per_day.iloc[i]), int(np.median(nb))))
    if sus:
        print("⚠ дні з різким провалом кількості тікерів (день, є, типово) — свято або дірка фетчу:")
        for d, v, m in sus:
            print(f"   {d}: {v:,} vs ~{m:,}")
        problems.append(f"{len(sus)} днів з провалом покриття")

    med = a[a["rth"] > 0].groupby("ticker")["rth"].median()
    dense = set(med[med >= min_rth_rows].index)
    dw = wk[wk["ticker"].isin(dense) & (wk["rth"] > 0)]
    # «торговий день» = хоч alive_share від максимуму тікерів із сесійними барами (свята/вихідні
    # відпадають, а тримісячна дірка половини універсу — ні, тож вона не маскується)
    sess_n = wk[wk["rth"] > 0].groupby("date")["ticker"].nunique()
    alive = set(sess_n[sess_n >= alive_share * sess_n.max()].index)
    exp_rth = dw.groupby("date")["rth"].median()
    have = set(zip(dw["ticker"], dw["date"]))
    span = dw.groupby("ticker")["date"].agg(["min", "max"])
    gaps = [(tk, d) for tk, r in span.iterrows() for d in days
            if r["min"] < d < r["max"] and d in alive and (tk, d) not in have]
    short = dw[dw["rth"] < 0.95 * dw["date"].map(exp_rth)]
    gdf = pd.DataFrame(gaps, columns=["ticker", "date"])
    print(f"щільних тікерів: {len(dense):,} | торгових днів: {len(alive)}")
    print(f"повністю пропущених днів усередині історії щільних тікерів: {len(gdf):,}")
    if len(gdf):
        print(gdf.groupby("date").size().to_string())
        print("приклад:", gdf.head(10).values.tolist())
        problems.append("пропущені дні")
    print(f"щільних тікер-днів із неповною сесією (<95% медіани дня): {len(short):,}")
    if len(short):
        print(short.sort_values("rth").head(10)[["ticker", "date", "rth"]].to_string(index=False))
        problems.append("неповні сесії")
    print("✅ аудит чистий" if not problems else "❌ проблеми: " + "; ".join(problems))
    if problems and raise_on_fail:
        raise AssertionError("; ".join(problems))
    return {"problems": problems, "missing_days": gdf, "short_sessions": short, "suspicious_days": sus}


audit_report = audit_final_coverage(str(FINAL_PATH))


рядків: 111,611,206 | null у o/h/l/c/v: 0 | порушень OHLC: 0 | дублів (ticker,dt): 0
⚠ дні з різким провалом кількості тікерів (день, є, типово) — свято або дірка фетчу:
   2026-08-11: 5,337 vs ~6,862
   2026-08-12: 5,336 vs ~6,861
   2026-08-13: 5,345 vs ~6,845
   2026-09-07: 60 vs ~7,317


щільних тікерів: 1,031 | торгових днів: 64
повністю пропущених днів усередині історії щільних тікерів: 2,756
date
2026-08-11    326
2026-08-12    325
2026-08-13    325
2026-08-14     84
2026-08-17    184
2026-08-18    186
2026-08-19    167
2026-08-20    167
2026-08-21     93
2026-08-24     86
2026-08-25     59
2026-08-26     98
2026-08-27    123
2026-08-28    105
2026-09-15    161
2026-09-16    122
2026-09-17    119
2026-09-18     26
приклад: [['AGNC', '2026-08-11'], ['AGNC', '2026-08-12'], ['AGNC', '2026-08-13'], ['AGQ', '2026-08-11'], ['AGQ', '2026-08-12'], ['AGQ', '2026-08-13'], ['AI', '2026-08-11'], ['AI', '2026-08-12'], ['AI', '2026-08-13'], ['AIG', '2026-08-11']]
щільних тікер-днів із неповною сесією (<95% медіани дня): 2,248
ticker       date  rth
  CYPH 2026-08-10  208
  CYPH 2026-08-06  214
  CYPH 2026-08-04  231
  CYPH 2026-08-14  250
  CYPH 2026-08-13  255
SKYD/W 2026-10-13  257
  CYPH 2026-08-11  271
  CYPH 2026-08-07  274
  CYPH 2026-08-12  276
  CYPH 2026-08-05  283
❌ про